# 01 — Cohort assembly and sample QC (Sections 3.1–3.2)

In [ ]:
import os
from pathlib import Path

# Folder that holds the downloaded public data; all outputs are written into it.
# Set the environment variable GBM_DATA_DIR, or edit the default below.
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
print("Data folder:", BASE)

### STEP 1 — Locate Combined_RNA

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import os

# ============================================================
# STEP 1 — Locate Combined_RNA
# ============================================================

RNA_DIR = BASE / "Combined_RNA"

print("RNA directory:")
print(RNA_DIR)

if not RNA_DIR.exists():
    raise FileNotFoundError(f"Directory not found: {RNA_DIR}")


# ============================================================
# STEP 2 — Find all files recursively
# ============================================================

all_files = [f for f in RNA_DIR.rglob("*") if f.is_file()]

print("\n======================================")
print("DIRECTORY SUMMARY")
print("======================================")

print("Total files:", len(all_files))


# ============================================================
# STEP 3 — Count file extensions
# ============================================================

extensions = Counter()

for f in all_files:
    extensions[f.suffix.lower()] += 1

print("\nFile extensions:")
for ext, count in extensions.most_common():
    print(f"{ext or '[no extension]'}: {count}")


# ============================================================
# STEP 4 — Show first 20 files
# ============================================================

print("\n======================================")
print("FIRST 20 FILES")
print("======================================")

for f in all_files[:20]:
    print(f.relative_to(RNA_DIR))


# ============================================================
# STEP 5 — File sizes
# ============================================================

sizes_mb = [(f, f.stat().st_size / (1024**2)) for f in all_files]

sizes_mb = sorted(
    sizes_mb,
    key=lambda x: x[1],
    reverse=True
)

print("\n======================================")
print("10 LARGEST FILES")
print("======================================")

for f, size in sizes_mb[:10]:
    print(f"{size:.2f} MB   {f.relative_to(RNA_DIR)}")


# ============================================================
# STEP 6 — Find likely RNA expression files
# ============================================================

rna_files = []

for f in all_files:

    name = f.name.lower()

    if (
        name.endswith(".tsv")
        or name.endswith(".txt")
        or name.endswith(".csv")
        or name.endswith(".tsv.gz")
        or "gene_counts" in name
        or "rna_seq" in name
    ):
        rna_files.append(f)


print("\n======================================")
print("POSSIBLE RNA EXPRESSION FILES")
print("======================================")

print("Possible RNA files:", len(rna_files))

for f in rna_files[:10]:
    print(f.relative_to(RNA_DIR))


# ============================================================
# STEP 7 — Inspect first RNA file
# ============================================================

if len(rna_files) > 0:

    first_file = rna_files[0]

    print("\n======================================")
    print("INSPECTING FIRST RNA FILE")
    print("======================================")

    print("File:")
    print(first_file)

    try:

        # GDC RNA files are normally TSV
        df = pd.read_csv(
            first_file,
            sep="\t",
            nrows=10,
            comment="#"
        )

        print("\nShape of preview:")
        print(df.shape)

        print("\nColumns:")
        print(df.columns.tolist())

        print("\nFirst 10 rows:")
        print(df.head(10).to_string())

    except Exception as e:

        print("\nCould not automatically read file.")
        print("Error:")
        print(e)


# ============================================================
# STEP 8 — Look for GDC metadata/sample sheets
# ============================================================

print("\n======================================")
print("METADATA FILES")
print("======================================")

metadata_keywords = [
    "sample_sheet",
    "sample-sheet",
    "metadata",
    "manifest",
    "cart"
]

metadata_files = []

for f in all_files:

    name = f.name.lower()

    if any(keyword in name for keyword in metadata_keywords):
        metadata_files.append(f)


if metadata_files:
    for f in metadata_files:
        print(f.relative_to(RNA_DIR))
else:
    print("No obvious metadata/sample sheet found inside Combined_RNA.")

### STEP 1 — Locate RNA files

In [ ]:
from pathlib import Path
import pandas as pd
import requests
import time

# ============================================================
# STEP 1 — Locate RNA files
# ============================================================

RNA_DIR = BASE / "Combined_RNA"

rna_files = list(RNA_DIR.rglob("*.tsv"))

file_names = [f.name for f in rna_files]

print("RNA files:", len(file_names))


# ============================================================
# STEP 2 — Query file-specific GDC associations
# ============================================================

endpoint = "https://api.gdc.cancer.gov/files"

fields = [
    "file_id",
    "file_name",
    "data_type",
    "experimental_strategy",
    "analysis.workflow_type",
    "cases.submitter_id",
    "cases.project.project_id",
    "associated_entities.entity_id",
    "associated_entities.entity_submitter_id",
    "associated_entities.entity_type"
]

all_hits = []

BATCH_SIZE = 100

for start in range(0, len(file_names), BATCH_SIZE):

    batch = file_names[start:start + BATCH_SIZE]

    filters = {
        "op": "in",
        "content": {
            "field": "files.file_name",
            "value": batch
        }
    }

    payload = {
        "filters": filters,
        "fields": ",".join(fields),
        "format": "JSON",
        "size": 1000
    }

    print(
        f"Querying {start + 1} to "
        f"{min(start + BATCH_SIZE, len(file_names))}"
    )

    response = requests.post(
        endpoint,
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    hits = response.json()["data"]["hits"]

    all_hits.extend(hits)

    print("Returned:", len(hits))

    time.sleep(0.2)


print("\nTotal file records:", len(all_hits))


# ============================================================
# STEP 3 — Build one row per file / associated entity
# ============================================================

rows = []

for hit in all_hits:

    file_id = hit.get("file_id")
    file_name = hit.get("file_name")

    cases = hit.get("cases", [])

    patient_id = None
    project_id = None

    if cases:
        patient_id = cases[0].get("submitter_id")

        if isinstance(cases[0].get("project"), dict):
            project_id = cases[0]["project"].get("project_id")

    entities = hit.get("associated_entities", [])

    if not entities:

        rows.append({
            "File ID": file_id,
            "File Name": file_name,
            "Patient ID": patient_id,
            "Project ID": project_id,
            "Associated Entity Type": None,
            "Associated Entity ID": None,
            "Associated Entity Barcode": None
        })

    else:

        for entity in entities:

            rows.append({
                "File ID": file_id,
                "File Name": file_name,
                "Patient ID": patient_id,
                "Project ID": project_id,
                "Associated Entity Type":
                    entity.get("entity_type"),

                "Associated Entity ID":
                    entity.get("entity_id"),

                "Associated Entity Barcode":
                    entity.get("entity_submitter_id")
            })


assoc = pd.DataFrame(rows)


# ============================================================
# STEP 4 — Examine entity types
# ============================================================

print("\n======================================")
print("ASSOCIATED ENTITY TYPES")
print("======================================")

print(
    assoc["Associated Entity Type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# STEP 5 — Extract TCGA sample code from barcode
# ============================================================

def extract_sample_code(barcode):

    if pd.isna(barcode):
        return None

    barcode = str(barcode)

    parts = barcode.split("-")

    # TCGA-XX-XXXX-01A-...
    if len(parts) >= 4 and barcode.startswith("TCGA-"):

        sample_part = parts[3]

        if len(sample_part) >= 2:
            return sample_part[:2]

    return None


assoc["Sample Type Code"] = (
    assoc["Associated Entity Barcode"]
    .apply(extract_sample_code)
)


# ============================================================
# STEP 6 — Convert sample code to biological type
# ============================================================

sample_code_map = {
    "01": "Primary Solid Tumor",
    "02": "Recurrent Solid Tumor",
    "03": "Primary Blood Derived Cancer",
    "05": "Additional New Primary",
    "06": "Metastatic",
    "10": "Blood Derived Normal",
    "11": "Solid Tissue Normal"
}

assoc["Sample Type Derived"] = (
    assoc["Sample Type Code"]
    .map(sample_code_map)
)


print("\n======================================")
print("FILE-SPECIFIC SAMPLE TYPES")
print("======================================")

print(
    assoc["Sample Type Derived"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# STEP 7 — Check how many unique files
# ============================================================

print("\n======================================")
print("UNIQUE FILES")
print("======================================")

print("Files represented:", assoc["File ID"].nunique())


# ============================================================
# STEP 8 — Keep primary tumor RNA
# ============================================================

primary = assoc[
    assoc["Sample Type Code"].eq("01")
].copy()


print("\n======================================")
print("TRUE PRIMARY TUMOR RNA")
print("======================================")

print(
    "Primary tumor RNA files:",
    primary["File ID"].nunique()
)

print(
    "Unique patients with primary RNA:",
    primary["Patient ID"].nunique()
)


# ============================================================
# STEP 9 — Count RNA files per patient
# ============================================================

files_per_patient = (
    primary
    .groupby("Patient ID")["File ID"]
    .nunique()
)

print("\n======================================")
print("PRIMARY RNA FILES PER PATIENT")
print("======================================")

print(
    files_per_patient
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    "\nPatients with more than one primary RNA file:",
    int((files_per_patient > 1).sum())
)


# ============================================================
# STEP 10 — Save
# ============================================================

OUT = BASE

assoc.to_csv(
    OUT / "Combined_RNA_file_specific_metadata.csv",
    index=False
)

primary.to_csv(
    OUT / "Combined_RNA_primary_tumor_files.csv",
    index=False
)

print("\nSaved:")
print(OUT / "Combined_RNA_file_specific_metadata.csv")
print(OUT / "Combined_RNA_primary_tumor_files.csv")

### STEP 1 — Downloads folder

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# STEP 1 — Downloads folder
# ============================================================

DOWNLOADS = BASE

# RNA metadata created in the previous step
RNA_METADATA = DOWNLOADS / "Combined_RNA_correct_primary_mapping.csv"

# Our cleaned clinical cohort
CLINICAL_FILE = DOWNLOADS / "TCGA_GBM_stage1D_IDH_MAF_annotated.tsv"

# Final IDH audit
IDH_FILE = DOWNLOADS / "TCGA_GBM_stage1E_IDH_source_audit.tsv"


# ============================================================
# STEP 2 — Check files
# ============================================================

for f in [RNA_METADATA, CLINICAL_FILE, IDH_FILE]:

    print(f)

    if not f.exists():
        raise FileNotFoundError(f"Missing file: {f}")


# ============================================================
# STEP 3 — Load files
# ============================================================

rna = pd.read_csv(
    RNA_METADATA,
    low_memory=False
)

clinical = pd.read_csv(
    CLINICAL_FILE,
    sep="\t",
    low_memory=False
)

idh = pd.read_csv(
    IDH_FILE,
    sep="\t",
    low_memory=False
)


print("\nRNA metadata rows:", len(rna))
print("Clinical patients:", clinical["Patient ID"].nunique())
print("IDH patients:", idh["Patient ID"].nunique())


# ============================================================
# STEP 4 — Create UNIQUE RNA patient list
# ============================================================

rna_patients = (
    rna[
        ["Patient ID"]
    ]
    .dropna()
    .drop_duplicates()
)

print("\n======================================")
print("RNA PATIENTS")
print("======================================")

print(
    "Unique primary-tumour RNA patients:",
    len(rna_patients)
)


# ============================================================
# STEP 5 — Add final IDH classification to clinical cohort
# ============================================================

idh_small = (
    idh[
        [
            "Patient ID",
            "Final_IDH_status",
            "IDH_evidence"
        ]
    ]
    .drop_duplicates("Patient ID")
)


cohort = clinical.merge(
    idh_small,
    on="Patient ID",
    how="left"
)


# ============================================================
# STEP 6 — Keep clinical patients who have RNA
# ============================================================

rna_patient_set = set(rna_patients["Patient ID"])

cohort["RNA_available"] = (
    cohort["Patient ID"].isin(rna_patient_set)
)


overlap = cohort[
    cohort["RNA_available"]
].copy()


print("\n======================================")
print("CLINICAL + RNA OVERLAP")
print("======================================")

print(
    "Clean clinical cohort:",
    cohort["Patient ID"].nunique()
)

print(
    "Patients with primary RNA:",
    overlap["Patient ID"].nunique()
)


# ============================================================
# STEP 7 — IDH distribution
# ============================================================

print("\n======================================")
print("IDH STATUS AMONG RNA PATIENTS")
print("======================================")

print(
    overlap["Final_IDH_status"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# STEP 8 — Separate groups
# ============================================================

idh_wt = overlap[
    overlap["Final_IDH_status"]
    .eq("IDH_WT_CONFIRMED")
].copy()


idh_mut = overlap[
    overlap["Final_IDH_status"]
    .eq("IDH_MUTANT_CONFIRMED")
].copy()


idh_unknown = overlap[
    overlap["Final_IDH_status"]
    .eq("IDH_UNRESOLVED")
].copy()


idh_conflict = overlap[
    overlap["Final_IDH_status"]
    .eq("IDH_CONFLICT")
].copy()


# ============================================================
# STEP 9 — Survival summary for IDH-WT patients
# ============================================================

print("\n======================================")
print("CONFIRMED IDH-WT + RNA")
print("======================================")

print(
    "Patients:",
    idh_wt["Patient ID"].nunique()
)


# Try to find OS fields
possible_os_time = [
    "Overall Survival (Months)",
    "OS_months",
    "OS"
]

possible_os_status = [
    "Overall Survival Status",
    "OS_status",
    "vital_status"
]


os_time_col = None
os_status_col = None

for col in possible_os_time:
    if col in idh_wt.columns:
        os_time_col = col
        break

for col in possible_os_status:
    if col in idh_wt.columns:
        os_status_col = col
        break


print("\nSurvival time column:", os_time_col)
print("Survival status column:", os_status_col)


if os_time_col is not None:

    os_numeric = pd.to_numeric(
        idh_wt[os_time_col],
        errors="coerce"
    )

    print(
        "Patients with OS time:",
        os_numeric.notna().sum()
    )

    print(
        "Median OS:",
        round(os_numeric.median(), 2),
        "months"
    )


if os_status_col is not None:

    print("\nSurvival status distribution:")

    print(
        idh_wt[os_status_col]
        .value_counts(dropna=False)
        .to_string()
    )


# ============================================================
# STEP 10 — Summary table
# ============================================================

summary = pd.DataFrame({

    "Group": [

        "Clean clinical cohort",

        "Primary RNA patients",

        "Clinical + RNA overlap",

        "Confirmed IDH-WT + RNA",

        "Confirmed IDH-mutant + RNA",

        "Unresolved IDH + RNA",

        "IDH conflict + RNA"
    ],

    "N": [

        cohort["Patient ID"].nunique(),

        rna_patients["Patient ID"].nunique(),

        overlap["Patient ID"].nunique(),

        idh_wt["Patient ID"].nunique(),

        idh_mut["Patient ID"].nunique(),

        idh_unknown["Patient ID"].nunique(),

        idh_conflict["Patient ID"].nunique()
    ]
})


print("\n======================================")
print("FINAL SUMMARY")
print("======================================")

print(summary.to_string(index=False))


# ============================================================
# STEP 11 — Save
# ============================================================

overlap.to_csv(
    DOWNLOADS / "TCGA_GBM_RNA_clinical_overlap.tsv",
    sep="\t",
    index=False
)


idh_wt.to_csv(
    DOWNLOADS / "TCGA_GBM_IDHWT_RNA_cohort.tsv",
    sep="\t",
    index=False
)


idh_unknown.to_csv(
    DOWNLOADS / "TCGA_GBM_RNA_IDH_unresolved.tsv",
    sep="\t",
    index=False
)


summary.to_csv(
    DOWNLOADS / "TCGA_GBM_RNA_cohort_summary.csv",
    index=False
)


print("\nFiles saved in Downloads.")

In [ ]:
from pathlib import Path
import pandas as pd

DOWNLOADS = BASE

RNA_META = DOWNLOADS / "TCGA_GBM_REAL_RNA_primary_metadata.csv"
COHORT = DOWNLOADS / "TCGA_GBM_REAL_IDHWT_RNA_cohort.tsv"

rna = pd.read_csv(RNA_META, low_memory=False)
cohort = pd.read_csv(COHORT, sep="\t", low_memory=False)

# Keep only our final 144 IDH-WT patients
patients144 = set(cohort["Patient ID"].dropna())

rna144 = rna[
    rna["Patient ID"].isin(patients144)
].copy()

# Count RNA files per patient
counts = (
    rna144.groupby("Patient ID")["File ID"]
    .nunique()
)

duplicates = counts[counts > 1].index

dup_table = rna144[
    rna144["Patient ID"].isin(duplicates)
].copy()

print("======================================")
print("FINAL 144-PATIENT RNA COHORT")
print("======================================")

print("Patients:", rna144["Patient ID"].nunique())
print("RNA files:", rna144["File ID"].nunique())

print("\n======================================")
print("DUPLICATE PATIENTS")
print("======================================")

print("Patients with >1 RNA file:", len(duplicates))

cols = [
    "Patient ID",
    "File ID",
    "File Name",
    "Aliquot Barcode",
    "Sample Code",
    "Sample Type"
]

print(
    dup_table[cols]
    .sort_values(["Patient ID", "Aliquot Barcode"])
    .to_string(index=False)
)

# Save for inspection
dup_table[cols].to_csv(
    DOWNLOADS / "TCGA_GBM_144_RNA_duplicate_patients.csv",
    index=False
)

print("\nSaved:")
print(
    DOWNLOADS / "TCGA_GBM_144_RNA_duplicate_patients.csv"
)

### TCGA-GBM

In [ ]:
# ============================================================
# TCGA-GBM
# Compare duplicate RNA-seq files for the 144-patient cohort
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# STEP 1 — PATHS
# ============================================================

DOWNLOADS = BASE

RNA_DIR = DOWNLOADS / "Combined_RNA"

DUP_FILE = (
    DOWNLOADS /
    "TCGA_GBM_144_RNA_duplicate_patients.csv"
)


# ============================================================
# STEP 2 — CHECK PATHS
# ============================================================

print("RNA directory:")
print(RNA_DIR)

print("\nDuplicate-patient file:")
print(DUP_FILE)


if not RNA_DIR.exists():
    raise FileNotFoundError(
        f"RNA directory not found: {RNA_DIR}"
    )


if not DUP_FILE.exists():
    raise FileNotFoundError(
        f"Duplicate file not found: {DUP_FILE}"
    )


# ============================================================
# STEP 3 — LOAD DUPLICATE PATIENT TABLE
# ============================================================

dup = pd.read_csv(
    DUP_FILE,
    low_memory=False
)


print("\n======================================")
print("DUPLICATE INPUT")
print("======================================")

print(
    "Duplicate patients:",
    dup["Patient ID"].nunique()
)

print(
    "RNA files involved:",
    dup["File Name"].nunique()
)


# ============================================================
# STEP 4 — FUNCTION TO FIND RNA FILE
# ============================================================

def find_rna_file(file_name):

    # First assume file is directly inside Combined_RNA
    direct_path = RNA_DIR / file_name

    if direct_path.exists():
        return direct_path

    # Otherwise search subfolders
    matches = list(
        RNA_DIR.rglob(file_name)
    )

    if len(matches) == 0:

        raise FileNotFoundError(
            f"RNA file not found: {file_name}"
        )

    if len(matches) > 1:

        print(
            f"WARNING: multiple copies found for {file_name}"
        )

    return matches[0]


# ============================================================
# STEP 5 — FUNCTION TO LOAD ONE GDC RNA FILE
# ============================================================

def load_rna(file_name):

    path = find_rna_file(file_name)

    # IMPORTANT:
    # GDC STAR files contain metadata lines beginning with #
    df = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        low_memory=False
    )

    # Required RNA columns
    required_columns = {
        "gene_id",
        "gene_name",
        "gene_type",
        "unstranded",
        "tpm_unstranded"
    }

    missing = (
        required_columns -
        set(df.columns)
    )

    if missing:

        raise ValueError(
            f"\nProblem with file:\n{file_name}\n\n"
            f"Columns found:\n{df.columns.tolist()}\n\n"
            f"Missing columns:\n{missing}"
        )

    # -----------------------------------------
    # Remove STAR summary rows
    # -----------------------------------------
    #
    # For example:
    # N_unmapped
    # N_multimapping
    # N_noFeature
    # N_ambiguous
    #
    # Actual genes begin with ENSG
    # -----------------------------------------

    genes = df[
        df["gene_id"]
        .astype(str)
        .str.startswith("ENSG")
    ].copy()

    # Keep only columns required for this analysis
    genes = genes[
        [
            "gene_id",
            "gene_name",
            "gene_type",
            "unstranded",
            "tpm_unstranded"
        ]
    ].copy()

    # Convert expression columns to numeric
    genes["unstranded"] = pd.to_numeric(
        genes["unstranded"],
        errors="coerce"
    ).fillna(0)

    genes["tpm_unstranded"] = pd.to_numeric(
        genes["tpm_unstranded"],
        errors="coerce"
    ).fillna(0)

    return genes


# ============================================================
# STEP 6 — TEST ONE FILE BEFORE ANALYZING EVERYTHING
# ============================================================

test_file = (
    dup["File Name"]
    .drop_duplicates()
    .iloc[0]
)

print("\n======================================")
print("TESTING ONE RNA FILE")
print("======================================")

print("File:")
print(test_file)

test = load_rna(test_file)

print("\nRNA table shape:")
print(test.shape)

print("\nColumns:")
print(test.columns.tolist())

print("\nFirst 5 genes:")
print(
    test.head().to_string(index=False)
)


# ============================================================
# STEP 7 — CHECK NUMBER OF FILES PER DUPLICATE PATIENT
# ============================================================

file_counts = (
    dup
    .groupby("Patient ID")["File Name"]
    .nunique()
)


print("\n======================================")
print("FILES PER DUPLICATE PATIENT")
print("======================================")

print(
    file_counts
    .value_counts()
    .sort_index()
    .to_string()
)


patients_not_two = (
    file_counts[
        file_counts != 2
    ]
)


if len(patients_not_two) > 0:

    print(
        "\nWARNING: These patients do not have exactly 2 RNA files:"
    )

    print(
        patients_not_two.to_string()
    )


# ============================================================
# STEP 8 — COMPARE THE TWO RNA FILES FOR EACH PATIENT
# ============================================================

results = []


for patient, group in dup.groupby("Patient ID"):

    file_names = (
        group["File Name"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    # We expect two RNA files per duplicate patient
    if len(file_names) != 2:

        print(
            f"Skipping {patient}: "
            f"{len(file_names)} files found."
        )

        continue


    file1 = file_names[0]
    file2 = file_names[1]


    print(
        f"\nProcessing {patient}..."
    )


    # ========================================================
    # LOAD BOTH RNA FILES
    # ========================================================

    rna1 = load_rna(file1)
    rna2 = load_rna(file2)


    # ========================================================
    # MATCH SAME GENES
    # ========================================================

    merged = rna1.merge(
        rna2,
        on="gene_id",
        how="inner",
        suffixes=("_1", "_2")
    )


    # ========================================================
    # EXPRESSION VECTORS
    # ========================================================

    tpm1 = (
        merged["tpm_unstranded_1"]
        .astype(float)
    )

    tpm2 = (
        merged["tpm_unstranded_2"]
        .astype(float)
    )


    # log2(TPM + 1)
    log_tpm1 = np.log2(
        tpm1 + 1
    )

    log_tpm2 = np.log2(
        tpm2 + 1
    )


    # ========================================================
    # PEARSON CORRELATION
    # ========================================================

    pearson = log_tpm1.corr(
        log_tpm2,
        method="pearson"
    )


    # ========================================================
    # SPEARMAN CORRELATION
    # ========================================================

    spearman = log_tpm1.corr(
        log_tpm2,
        method="spearman"
    )


    # ========================================================
    # EXPRESSION DIFFERENCE
    # ========================================================

    mean_abs_difference = (
        np.abs(
            log_tpm1 -
            log_tpm2
        )
        .mean()
    )


    # ========================================================
    # GENE-DETECTION QC
    # ========================================================

    detected_file1 = int(
        (tpm1 > 0).sum()
    )

    detected_file2 = int(
        (tpm2 > 0).sum()
    )


    genes_tpm1_file1 = int(
        (tpm1 >= 1).sum()
    )

    genes_tpm1_file2 = int(
        (tpm2 >= 1).sum()
    )


    # ========================================================
    # TOTAL ASSIGNED RAW COUNTS
    # ========================================================

    counts1 = (
        merged["unstranded_1"]
        .astype(float)
    )

    counts2 = (
        merged["unstranded_2"]
        .astype(float)
    )


    total_counts1 = counts1.sum()
    total_counts2 = counts2.sum()


    # ========================================================
    # GET ALIQUOT BARCODES
    # ========================================================

    barcode1_rows = group[
        group["File Name"] == file1
    ]["Aliquot Barcode"]

    barcode2_rows = group[
        group["File Name"] == file2
    ]["Aliquot Barcode"]


    barcode1 = (
        barcode1_rows.iloc[0]
        if len(barcode1_rows) > 0
        else None
    )

    barcode2 = (
        barcode2_rows.iloc[0]
        if len(barcode2_rows) > 0
        else None
    )


    # ========================================================
    # STORE RESULT
    # ========================================================

    results.append({

        "Patient ID":
            patient,

        "File 1":
            file1,

        "Aliquot 1":
            barcode1,

        "File 2":
            file2,

        "Aliquot 2":
            barcode2,

        "Genes compared":
            len(merged),

        "Pearson log2TPM":
            pearson,

        "Spearman log2TPM":
            spearman,

        "Mean absolute log2TPM difference":
            mean_abs_difference,

        "Detected genes file1":
            detected_file1,

        "Detected genes file2":
            detected_file2,

        "Genes TPM>=1 file1":
            genes_tpm1_file1,

        "Genes TPM>=1 file2":
            genes_tpm1_file2,

        "Assigned counts file1":
            total_counts1,

        "Assigned counts file2":
            total_counts2

    })


# ============================================================
# STEP 9 — CREATE RESULTS TABLE
# ============================================================

results = pd.DataFrame(
    results
)


print("\n======================================")
print("DUPLICATE RNA CORRELATIONS")
print("======================================")


display_columns = [

    "Patient ID",

    "Pearson log2TPM",

    "Spearman log2TPM",

    "Mean absolute log2TPM difference",

    "Detected genes file1",

    "Detected genes file2",

    "Genes TPM>=1 file1",

    "Genes TPM>=1 file2"
]


print(

    results[
        display_columns
    ]
    .sort_values(
        "Spearman log2TPM"
    )
    .to_string(
        index=False
    )

)


# ============================================================
# STEP 10 — CORRELATION SUMMARY
# ============================================================

print("\n======================================")
print("CORRELATION SUMMARY")
print("======================================")


print(

    results[
        [
            "Pearson log2TPM",
            "Spearman log2TPM",
            "Mean absolute log2TPM difference"
        ]
    ]
    .describe()
    .to_string()

)


# ============================================================
# STEP 11 — LOWEST-CORRELATION PAIR
# ============================================================

print("\n======================================")
print("LOWEST CORRELATION PATIENT")
print("======================================")


if len(results) > 0:

    lowest_index = (
        results["Spearman log2TPM"]
        .idxmin()
    )

    lowest = (
        results
        .loc[lowest_index]
    )


    print(
        lowest[
            [
                "Patient ID",

                "Aliquot 1",

                "Aliquot 2",

                "Pearson log2TPM",

                "Spearman log2TPM",

                "Mean absolute log2TPM difference",

                "Detected genes file1",

                "Detected genes file2",

                "Genes TPM>=1 file1",

                "Genes TPM>=1 file2",

                "Assigned counts file1",

                "Assigned counts file2"
            ]
        ]
        .to_string()
    )


# ============================================================
# STEP 12 — SIMPLE SIMILARITY CATEGORIES
# ============================================================

def correlation_category(x):

    if pd.isna(x):
        return "ERROR / MISSING"

    elif x >= 0.98:
        return "VERY HIGH"

    elif x >= 0.95:
        return "HIGH"

    elif x >= 0.90:
        return "MODERATE"

    else:
        return "LOW"


results["Similarity Category"] = (

    results["Spearman log2TPM"]
    .apply(
        correlation_category
    )

)


print("\n======================================")
print("SIMILARITY CATEGORIES")
print("======================================")


print(

    results[
        "Similarity Category"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()

)


# ============================================================
# STEP 13 — SAVE COMPLETE RESULTS
# ============================================================

OUTPUT_FILE = (

    DOWNLOADS /
    "TCGA_GBM_144_duplicate_RNA_correlations.csv"

)


results.to_csv(

    OUTPUT_FILE,

    index=False

)


print("\n======================================")
print("FINISHED")
print("======================================")


print(
    "Results saved to:"
)

print(
    OUTPUT_FILE
)

### PATHS

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# PATHS
# ============================================================

DOWNLOADS = BASE
RNA_DIR = DOWNLOADS / "Combined_RNA"

DUP_FILE = (
    DOWNLOADS /
    "TCGA_GBM_144_RNA_duplicate_patients.csv"
)

dup = pd.read_csv(DUP_FILE)


# ============================================================
# LOAD RNA
# ============================================================

def load_rna(file_name):

    path = RNA_DIR / file_name

    if not path.exists():
        matches = list(RNA_DIR.rglob(file_name))

        if len(matches) == 0:
            raise FileNotFoundError(file_name)

        path = matches[0]

    df = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        low_memory=False
    )

    df = df[
        df["gene_id"]
        .astype(str)
        .str.startswith("ENSG")
    ].copy()

    df["tpm_unstranded"] = pd.to_numeric(
        df["tpm_unstranded"],
        errors="coerce"
    ).fillna(0)

    df["unstranded"] = pd.to_numeric(
        df["unstranded"],
        errors="coerce"
    ).fillna(0)

    return df


# ============================================================
# COMPARE DUPLICATES
# ============================================================

results = []

for patient, group in dup.groupby("Patient ID"):

    files = (
        group["File Name"]
        .drop_duplicates()
        .tolist()
    )

    if len(files) != 2:
        continue

    f1, f2 = files

    x = load_rna(f1)
    y = load_rna(f2)

    # ------------------------------------------
    # Protein-coding genes only
    # ------------------------------------------

    x = x[
        x["gene_type"] == "protein_coding"
    ].copy()

    y = y[
        y["gene_type"] == "protein_coding"
    ].copy()

    m = x.merge(
        y,
        on="gene_id",
        suffixes=("_1", "_2")
    )

    # ------------------------------------------
    # Keep genes expressed in either aliquot
    # TPM >= 1
    # ------------------------------------------

    expressed = m[
        (m["tpm_unstranded_1"] >= 1) |
        (m["tpm_unstranded_2"] >= 1)
    ].copy()

    a = np.log2(
        expressed["tpm_unstranded_1"] + 1
    )

    b = np.log2(
        expressed["tpm_unstranded_2"] + 1
    )

    pearson = a.corr(
        b,
        method="pearson"
    )

    spearman = a.corr(
        b,
        method="spearman"
    )

    # ------------------------------------------
    # QC for choosing one file later
    # ------------------------------------------

    pc1 = x[
        x["tpm_unstranded"] >= 1
    ].shape[0]

    pc2 = y[
        y["tpm_unstranded"] >= 1
    ].shape[0]

    counts1 = x["unstranded"].sum()
    counts2 = y["unstranded"].sum()

    results.append({

        "Patient ID": patient,

        "File 1": f1,
        "File 2": f2,

        "Protein coding genes compared":
            len(expressed),

        "Pearson protein-coding":
            pearson,

        "Spearman protein-coding":
            spearman,

        "Protein genes TPM>=1 file1":
            pc1,

        "Protein genes TPM>=1 file2":
            pc2,

        "Protein assigned counts file1":
            counts1,

        "Protein assigned counts file2":
            counts2
    })


results = pd.DataFrame(results)


# ============================================================
# RESULTS
# ============================================================

print("\n======================================")
print("PROTEIN-CODING DUPLICATE CORRELATIONS")
print("======================================")

print(
    results[
        [
            "Patient ID",
            "Pearson protein-coding",
            "Spearman protein-coding",
            "Protein genes TPM>=1 file1",
            "Protein genes TPM>=1 file2"
        ]
    ]
    .sort_values("Spearman protein-coding")
    .to_string(index=False)
)


print("\n======================================")
print("SUMMARY")
print("======================================")

print(
    results[
        [
            "Pearson protein-coding",
            "Spearman protein-coding"
        ]
    ]
    .describe()
    .to_string()
)


results.to_csv(
    DOWNLOADS /
    "TCGA_GBM_duplicate_protein_coding_QC.csv",
    index=False
)

print("\nSaved:")
print(
    DOWNLOADS /
    "TCGA_GBM_duplicate_protein_coding_QC.csv"
)

### TCGA-GBM

In [ ]:
# ============================================================
# TCGA-GBM
# Select ONE RNA-seq aliquot per patient
# Final target: 144 patients = 144 RNA files
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# STEP 1 — PATHS
# ============================================================

DOWNLOADS = BASE

RNA_DIR = DOWNLOADS / "Combined_RNA"

RNA_META = (
    DOWNLOADS /
    "TCGA_GBM_REAL_RNA_primary_metadata.csv"
)

COHORT_FILE = (
    DOWNLOADS /
    "TCGA_GBM_REAL_IDHWT_RNA_cohort.tsv"
)


# ============================================================
# STEP 2 — LOAD DATA
# ============================================================

rna_meta = pd.read_csv(
    RNA_META,
    low_memory=False
)

cohort = pd.read_csv(
    COHORT_FILE,
    sep="\t",
    low_memory=False
)


print("\n======================================")
print("INPUT")
print("======================================")

print(
    "IDH-WT patients:",
    cohort["Patient ID"].nunique()
)


# ============================================================
# STEP 3 — KEEP ONLY OUR 144 PATIENTS
# ============================================================

patient_set = set(
    cohort["Patient ID"]
    .dropna()
    .unique()
)

rna144 = rna_meta[
    rna_meta["Patient ID"]
    .isin(patient_set)
].copy()


# One row per physical RNA file
rna144 = (
    rna144
    .drop_duplicates(
        subset=["File ID"]
    )
    .copy()
)


print(
    "RNA files before duplicate resolution:",
    rna144["File ID"].nunique()
)

print(
    "Patients:",
    rna144["Patient ID"].nunique()
)


# ============================================================
# STEP 4 — FIND RNA FILE ON DISK
# ============================================================

def find_file(file_name):

    direct = RNA_DIR / file_name

    if direct.exists():
        return direct

    matches = list(
        RNA_DIR.rglob(file_name)
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Cannot find {file_name}"
        )

    return matches[0]


# ============================================================
# STEP 5 — PARSE ANALYTE TYPE FROM TCGA BARCODE
# ============================================================

def get_analyte_type(barcode):

    if pd.isna(barcode):
        return None

    parts = str(barcode).split("-")

    # Example:
    #
    # TCGA-06-0878-01A-01R-A96S-41
    #
    # parts[4] = 01R
    #
    # Final character = R

    if len(parts) >= 5:

        portion_analyte = parts[4]

        if len(portion_analyte) >= 1:
            return portion_analyte[-1]

    return None


rna144["Analyte Type"] = (
    rna144["Aliquot Barcode"]
    .apply(get_analyte_type)
)


print("\n======================================")
print("ANALYTE TYPES")
print("======================================")

print(
    rna144["Analyte Type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# STEP 6 — FUNCTION TO CALCULATE QC FOR ONE RNA FILE
# ============================================================

def calculate_qc(file_name):

    path = find_file(file_name)

    df = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        low_memory=False
    )

    # Real genes only
    df = df[
        df["gene_id"]
        .astype(str)
        .str.startswith("ENSG")
    ].copy()

    # Protein coding genes only
    pc = df[
        df["gene_type"]
        .eq("protein_coding")
    ].copy()

    pc["tpm_unstranded"] = pd.to_numeric(
        pc["tpm_unstranded"],
        errors="coerce"
    ).fillna(0)

    pc["unstranded"] = pd.to_numeric(
        pc["unstranded"],
        errors="coerce"
    ).fillna(0)

    # -----------------------------------------
    # QC metric 1:
    # number of protein coding genes TPM >= 1
    # -----------------------------------------

    detected_pc = int(
        (pc["tpm_unstranded"] >= 1)
        .sum()
    )

    # -----------------------------------------
    # QC metric 2:
    # total protein coding assigned counts
    # -----------------------------------------

    total_pc_counts = float(
        pc["unstranded"].sum()
    )

    # -----------------------------------------
    # Additional information
    # -----------------------------------------

    detected_any_pc = int(
        (pc["tpm_unstranded"] > 0)
        .sum()
    )

    return {
        "Protein genes TPM>=1":
            detected_pc,

        "Protein genes TPM>0":
            detected_any_pc,

        "Protein assigned counts":
            total_pc_counts
    }


# ============================================================
# STEP 7 — CALCULATE QC FOR ALL 167 FILES
# ============================================================

qc_results = []

total_files = len(rna144)

for i, row in enumerate(
    rna144.itertuples(index=False),
    start=1
):

    file_name = getattr(
        row,
        "_1",
        None
    )

    # Safer way because column names contain spaces
    file_name = rna144.iloc[i - 1]["File Name"]

    patient = rna144.iloc[i - 1]["Patient ID"]

    print(
        f"[{i}/{total_files}] "
        f"{patient}"
    )

    qc = calculate_qc(
        file_name
    )

    qc_results.append({

        "File ID":
            rna144.iloc[i - 1]["File ID"],

        "File Name":
            file_name,

        "Patient ID":
            patient,

        "Aliquot Barcode":
            rna144.iloc[i - 1]["Aliquot Barcode"],

        "Analyte Type":
            rna144.iloc[i - 1]["Analyte Type"],

        "Protein genes TPM>=1":
            qc["Protein genes TPM>=1"],

        "Protein genes TPM>0":
            qc["Protein genes TPM>0"],

        "Protein assigned counts":
            qc["Protein assigned counts"]
    })


qc = pd.DataFrame(
    qc_results
)


# ============================================================
# STEP 8 — SELECT ONE FILE PER PATIENT
# ============================================================

selected_rows = []
audit_rows = []


for patient, group in qc.groupby(
    "Patient ID"
):

    group = group.copy()

    n_files = len(group)

    # -----------------------------------------
    # SINGLE RNA FILE
    # -----------------------------------------

    if n_files == 1:

        chosen = group.iloc[0]

        reason = (
            "Only primary RNA file available"
        )

    # -----------------------------------------
    # DUPLICATE RNA FILES
    # -----------------------------------------

    else:

        candidates = group.copy()

        # -------------------------------------
        # Prefer R analyte if both R and T occur
        # -------------------------------------

        if (
            "R" in candidates["Analyte Type"].values
            and
            candidates["Analyte Type"].nunique() > 1
        ):

            candidates = candidates[
                candidates["Analyte Type"]
                .eq("R")
            ].copy()

            analyte_reason = (
                "Preferred RNA analyte R"
            )

        else:

            analyte_reason = (
                "Same analyte type"
            )

        # -------------------------------------
        # Sort by technical quality
        # -------------------------------------

        candidates = candidates.sort_values(

            by=[
                "Protein genes TPM>=1",
                "Protein assigned counts",
                "File Name"
            ],

            ascending=[
                False,
                False,
                True
            ]

        )

        chosen = candidates.iloc[0]

        reason = (
            analyte_reason
            +
            "; selected highest "
            "protein-coding detection, "
            "then assigned counts"
        )


    # ========================================================
    # STORE CHOSEN FILE
    # ========================================================

    selected_rows.append({

        "Patient ID":
            patient,

        "Selected File ID":
            chosen["File ID"],

        "Selected File Name":
            chosen["File Name"],

        "Selected Aliquot":
            chosen["Aliquot Barcode"],

        "Analyte Type":
            chosen["Analyte Type"],

        "Protein genes TPM>=1":
            chosen["Protein genes TPM>=1"],

        "Protein genes TPM>0":
            chosen["Protein genes TPM>0"],

        "Protein assigned counts":
            chosen["Protein assigned counts"],

        "Number RNA files":
            n_files,

        "Selection reason":
            reason
    })


    # ========================================================
    # AUDIT ALL FILES
    # ========================================================

    for _, candidate in group.iterrows():

        audit_rows.append({

            "Patient ID":
                patient,

            "File ID":
                candidate["File ID"],

            "File Name":
                candidate["File Name"],

            "Aliquot Barcode":
                candidate["Aliquot Barcode"],

            "Analyte Type":
                candidate["Analyte Type"],

            "Protein genes TPM>=1":
                candidate["Protein genes TPM>=1"],

            "Protein assigned counts":
                candidate["Protein assigned counts"],

            "Selected":
                candidate["File ID"]
                == chosen["File ID"]
        })


# ============================================================
# STEP 9 — CREATE TABLES
# ============================================================

selected = pd.DataFrame(
    selected_rows
)

audit = pd.DataFrame(
    audit_rows
)


# ============================================================
# STEP 10 — VALIDATE
# ============================================================

print("\n======================================")
print("FINAL SELECTION")
print("======================================")

print(
    "Patients:",
    selected["Patient ID"].nunique()
)

print(
    "Selected RNA files:",
    selected["Selected File ID"].nunique()
)

print(
    "Patients with one original RNA file:",
    int(
        (
            selected["Number RNA files"] == 1
        ).sum()
    )
)

print(
    "Patients resolved from duplicates:",
    int(
        (
            selected["Number RNA files"] > 1
        ).sum()
    )
)


# Must be exactly 144
assert (
    selected["Patient ID"].nunique()
    == 144
), "ERROR: Final patient count is not 144."


assert (
    selected["Selected File ID"].nunique()
    == 144
), "ERROR: Final RNA file count is not 144."


# ============================================================
# STEP 11 — SHOW DUPLICATE DECISIONS
# ============================================================

duplicate_decisions = selected[
    selected["Number RNA files"] > 1
].copy()


print("\n======================================")
print("DUPLICATE SELECTIONS")
print("======================================")

print(

    duplicate_decisions[
        [
            "Patient ID",
            "Selected Aliquot",
            "Analyte Type",
            "Protein genes TPM>=1",
            "Protein assigned counts",
            "Selection reason"
        ]
    ]
    .sort_values(
        "Patient ID"
    )
    .to_string(
        index=False
    )

)


# ============================================================
# STEP 12 — MERGE WITH CLINICAL COHORT
# ============================================================

final_inventory = cohort.merge(

    selected,

    on="Patient ID",

    how="inner"

)


print("\n======================================")
print("FINAL DISCOVERY INVENTORY")
print("======================================")

print(
    "Patients:",
    final_inventory["Patient ID"]
    .nunique()
)

print(
    "RNA files:",
    final_inventory["Selected File ID"]
    .nunique()
)


# ============================================================
# STEP 13 — SAVE
# ============================================================

selected.to_csv(

    DOWNLOADS /
    "TCGA_GBM_144_selected_RNA_files.csv",

    index=False

)


audit.to_csv(

    DOWNLOADS /
    "TCGA_GBM_144_RNA_selection_audit.csv",

    index=False

)


final_inventory.to_csv(

    DOWNLOADS /
    "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.tsv",

    sep="\t",

    index=False

)


print("\n======================================")
print("FILES CREATED")
print("======================================")

print(
    DOWNLOADS /
    "TCGA_GBM_144_selected_RNA_files.csv"
)

print(
    DOWNLOADS /
    "TCGA_GBM_144_RNA_selection_audit.csv"
)

print(
    DOWNLOADS /
    "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.tsv"
)

### STAGE 2A

In [ ]:
# ============================================================
# STAGE 2A
# Build TCGA-GBM RNA expression matrices
# 144 IDH-WT patients
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# STEP 1 — PATHS
# ============================================================

DOWNLOADS = BASE

RNA_DIR = DOWNLOADS / "Combined_RNA"

SELECTION_FILE = (
    DOWNLOADS /
    "TCGA_GBM_144_selected_RNA_files.csv"
)

OUTPUT_DIR = (
    DOWNLOADS /
    "TCGA_GBM_STAGE2"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)


# ============================================================
# STEP 2 — LOAD SELECTED FILE INVENTORY
# ============================================================

selected = pd.read_csv(
    SELECTION_FILE,
    low_memory=False
)

print("\n======================================")
print("INPUT")
print("======================================")

print(
    "Patients:",
    selected["Patient ID"].nunique()
)

print(
    "Selected RNA files:",
    selected["Selected File ID"].nunique()
)


assert selected["Patient ID"].nunique() == 144
assert selected["Selected File ID"].nunique() == 144


# ============================================================
# STEP 3 — FIND FILE ON DISK
# ============================================================

def find_rna_file(file_name):

    direct = RNA_DIR / file_name

    if direct.exists():
        return direct

    matches = list(
        RNA_DIR.rglob(file_name)
    )

    if len(matches) == 0:

        raise FileNotFoundError(
            f"Cannot find RNA file:\n{file_name}"
        )

    return matches[0]


# ============================================================
# STEP 4 — READ FIRST FILE TO DEFINE GENE UNIVERSE
# ============================================================

first_file = selected.iloc[0]["Selected File Name"]

first_path = find_rna_file(
    first_file
)

first = pd.read_csv(
    first_path,
    sep="\t",
    comment="#",
    low_memory=False
)


# Keep actual genes only
first = first[
    first["gene_id"]
    .astype(str)
    .str.startswith("ENSG")
].copy()


print("\n======================================")
print("GENE UNIVERSE")
print("======================================")

print(
    "All genes in first RNA file:",
    len(first)
)

print("\nGene types:")

print(
    first["gene_type"]
    .value_counts()
    .head(20)
    .to_string()
)


# ============================================================
# STEP 5 — KEEP PROTEIN-CODING GENES
# ============================================================

first_pc = first[
    first["gene_type"].eq(
        "protein_coding"
    )
].copy()


print(
    "\nProtein-coding genes:",
    len(first_pc)
)


# Remove Ensembl version:
#
# ENSG00000000003.15
# becomes
# ENSG00000000003

first_pc["ensembl_id"] = (
    first_pc["gene_id"]
    .astype(str)
    .str.split(".")
    .str[0]
)


gene_annotation = (
    first_pc[
        [
            "ensembl_id",
            "gene_id",
            "gene_name",
            "gene_type"
        ]
    ]
    .drop_duplicates(
        "ensembl_id"
    )
    .copy()
)


# ============================================================
# STEP 6 — STORAGE FOR MATRICES
# ============================================================

tpm_data = {}
count_data = {}

qc_rows = []

reference_genes = set(
    gene_annotation["ensembl_id"]
)


# ============================================================
# STEP 7 — READ ALL 144 SELECTED RNA FILES
# ============================================================

total = len(selected)


for i, row in selected.iterrows():

    patient = row["Patient ID"]

    file_name = row[
        "Selected File Name"
    ]

    path = find_rna_file(
        file_name
    )


    print(
        f"[{i + 1}/{total}] "
        f"{patient}"
    )


    df = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        low_memory=False
    )


    # -----------------------------------------
    # Keep real genes
    # -----------------------------------------

    df = df[
        df["gene_id"]
        .astype(str)
        .str.startswith("ENSG")
    ].copy()


    # -----------------------------------------
    # Protein-coding genes only
    # -----------------------------------------

    df = df[
        df["gene_type"]
        .eq("protein_coding")
    ].copy()


    # -----------------------------------------
    # Standardize Ensembl ID
    # -----------------------------------------

    df["ensembl_id"] = (
        df["gene_id"]
        .astype(str)
        .str.split(".")
        .str[0]
    )


    # -----------------------------------------
    # Numeric expression
    # -----------------------------------------

    df["tpm_unstranded"] = pd.to_numeric(
        df["tpm_unstranded"],
        errors="coerce"
    ).fillna(0)


    df["unstranded"] = pd.to_numeric(
        df["unstranded"],
        errors="coerce"
    ).fillna(0)


    # -----------------------------------------
    # One row per Ensembl gene
    # -----------------------------------------

    df = (
        df
        .drop_duplicates(
            "ensembl_id"
        )
        .set_index(
            "ensembl_id"
        )
    )


    # -----------------------------------------
    # Check gene consistency
    # -----------------------------------------

    current_genes = set(
        df.index
    )

    missing_reference = len(
        reference_genes -
        current_genes
    )

    extra_genes = len(
        current_genes -
        reference_genes
    )


    # -----------------------------------------
    # Align to common reference
    # -----------------------------------------

    df = df.reindex(
        gene_annotation["ensembl_id"]
    )


    tpm = (
        df["tpm_unstranded"]
        .fillna(0)
    )


    counts = (
        df["unstranded"]
        .fillna(0)
    )


    # -----------------------------------------
    # Store column
    # -----------------------------------------

    tpm_data[patient] = (
        tpm.values
    )

    count_data[patient] = (
        counts.values
    )


    # -----------------------------------------
    # Basic QC
    # -----------------------------------------

    qc_rows.append({

        "Patient ID":
            patient,

        "Selected File Name":
            file_name,

        "Protein genes":
            len(df),

        "Detected genes TPM>0":
            int(
                (tpm > 0).sum()
            ),

        "Detected genes TPM>=1":
            int(
                (tpm >= 1).sum()
            ),

        "Total protein counts":
            float(
                counts.sum()
            ),

        "Median TPM":
            float(
                tpm.median()
            ),

        "Mean TPM":
            float(
                tpm.mean()
            ),

        "Missing reference genes":
            missing_reference,

        "Extra genes":
            extra_genes
    })


# ============================================================
# STEP 8 — BUILD MATRICES
# ============================================================

tpm_matrix = pd.DataFrame(
    tpm_data,
    index=gene_annotation["ensembl_id"]
)

count_matrix = pd.DataFrame(
    count_data,
    index=gene_annotation["ensembl_id"]
)


# rows = genes
# columns = patients


print("\n======================================")
print("FINAL MATRIX DIMENSIONS")
print("======================================")

print(
    "TPM matrix:",
    tpm_matrix.shape
)

print(
    "Count matrix:",
    count_matrix.shape
)


# ============================================================
# STEP 9 — CHECK FOR MISSING VALUES
# ============================================================

print("\n======================================")
print("MISSING VALUES")
print("======================================")

print(
    "TPM NaN:",
    int(
        tpm_matrix.isna().sum().sum()
    )
)

print(
    "Counts NaN:",
    int(
        count_matrix.isna().sum().sum()
    )
)


# ============================================================
# STEP 10 — DUPLICATE PATIENT CHECK
# ============================================================

print("\n======================================")
print("PATIENT CHECK")
print("======================================")

print(
    "TPM patient columns:",
    tpm_matrix.shape[1]
)

print(
    "Unique patient IDs:",
    tpm_matrix.columns.nunique()
)


assert tpm_matrix.shape[1] == 144
assert tpm_matrix.columns.nunique() == 144


# ============================================================
# STEP 11 — BASIC QC TABLE
# ============================================================

qc = pd.DataFrame(
    qc_rows
)


print("\n======================================")
print("SAMPLE QC SUMMARY")
print("======================================")

qc_columns = [

    "Detected genes TPM>0",

    "Detected genes TPM>=1",

    "Total protein counts",

    "Median TPM",

    "Mean TPM",

    "Missing reference genes",

    "Extra genes"
]


print(
    qc[
        qc_columns
    ]
    .describe()
    .to_string()
)


# ============================================================
# STEP 12 — IDENTIFY EXTREME SAMPLES
# ============================================================

print("\n======================================")
print("LOWEST GENE-DETECTION SAMPLES")
print("======================================")

print(

    qc[
        [
            "Patient ID",
            "Detected genes TPM>=1",
            "Total protein counts"
        ]
    ]
    .sort_values(
        "Detected genes TPM>=1"
    )
    .head(10)
    .to_string(
        index=False
    )

)


print("\n======================================")
print("LOWEST COUNT SAMPLES")
print("======================================")

print(

    qc[
        [
            "Patient ID",
            "Detected genes TPM>=1",
            "Total protein counts"
        ]
    ]
    .sort_values(
        "Total protein counts"
    )
    .head(10)
    .to_string(
        index=False
    )

)


# ============================================================
# STEP 13 — SAVE
# ============================================================

tpm_matrix.to_csv(

    OUTPUT_DIR /
    "TCGA_GBM_144_protein_coding_TPM_matrix.csv"

)


count_matrix.to_csv(

    OUTPUT_DIR /
    "TCGA_GBM_144_protein_coding_RAW_COUNTS_matrix.csv"

)


gene_annotation.to_csv(

    OUTPUT_DIR /
    "TCGA_GBM_gene_annotation.csv",

    index=False

)


qc.to_csv(

    OUTPUT_DIR /
    "TCGA_GBM_144_sample_QC.csv",

    index=False

)


print("\n======================================")
print("FILES CREATED")
print("======================================")

print(
    OUTPUT_DIR /
    "TCGA_GBM_144_protein_coding_TPM_matrix.csv"
)

print(
    OUTPUT_DIR /
    "TCGA_GBM_144_protein_coding_RAW_COUNTS_matrix.csv"
)

print(
    OUTPUT_DIR /
    "TCGA_GBM_gene_annotation.csv"
)

print(
    OUTPUT_DIR /
    "TCGA_GBM_144_sample_QC.csv"
)

### STAGE 2B

In [ ]:
# ============================================================
# STAGE 2B
# TCGA-GBM RNA-seq SAMPLE-LEVEL QC
#
# Input:
#   144 patients x 19,944 protein-coding genes
#
# Purpose:
#   Detect possible expression outliers BEFORE network analysis.
#
# IMPORTANT:
#   This PCA is QC only.
#   It is NOT survival feature selection.
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# ============================================================
# STEP 1 — PATHS
# ============================================================

DOWNLOADS = BASE

STAGE2 = DOWNLOADS / "TCGA_GBM_STAGE2"

TPM_FILE = (
    STAGE2 /
    "TCGA_GBM_144_protein_coding_TPM_matrix.csv"
)

QC_FILE = (
    STAGE2 /
    "TCGA_GBM_144_sample_QC.csv"
)

OUTPUT_DIR = (
    STAGE2 /
    "QC_STAGE2B"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)


# ============================================================
# STEP 2 — LOAD TPM MATRIX
# ============================================================

tpm = pd.read_csv(
    TPM_FILE,
    index_col=0
)

sample_qc = pd.read_csv(
    QC_FILE
)


print("\n======================================")
print("INPUT")
print("======================================")

print("Genes:", tpm.shape[0])
print("Patients:", tpm.shape[1])

print(
    "Missing values:",
    int(tpm.isna().sum().sum())
)


# ============================================================
# STEP 3 — LOG2(TPM + 1)
# ============================================================

log_tpm = np.log2(
    tpm + 1
)


print("\n======================================")
print("LOG TPM")
print("======================================")

print(
    "Minimum:",
    float(log_tpm.min().min())
)

print(
    "Maximum:",
    float(log_tpm.max().max())
)


# ============================================================
# STEP 4 — QC-ONLY GENE FILTER
# ============================================================
#
# For PCA only:
#
# Keep genes with TPM >= 1 in at least 20% of patients.
#
# This removes genes that are essentially absent everywhere.
#
# DO NOT confuse this with final model feature selection.
# ============================================================

minimum_patients = int(
    np.ceil(
        0.20 * tpm.shape[1]
    )
)

expressed_mask = (
    (tpm >= 1)
    .sum(axis=1)
    >= minimum_patients
)

qc_expression = log_tpm.loc[
    expressed_mask
].copy()


print("\n======================================")
print("QC GENE FILTER")
print("======================================")

print(
    "Minimum patients required:",
    minimum_patients
)

print(
    "Genes retained for QC:",
    qc_expression.shape[0]
)


# ============================================================
# STEP 5 — GENE VARIANCE
# ============================================================
#
# For PCA visualization we use the most variable genes.
# Again: QC only.
# ============================================================

gene_variance = (
    qc_expression
    .var(axis=1)
    .sort_values(
        ascending=False
    )
)

N_TOP_GENES = min(
    5000,
    len(gene_variance)
)

top_genes = (
    gene_variance
    .head(N_TOP_GENES)
    .index
)

pca_data = (
    qc_expression
    .loc[top_genes]
    .T
)


print("\n======================================")
print("PCA INPUT")
print("======================================")

print(
    "Patients:",
    pca_data.shape[0]
)

print(
    "Genes:",
    pca_data.shape[1]
)


# ============================================================
# STEP 6 — STANDARDIZE GENES FOR PCA
# ============================================================

scaler = StandardScaler()

X = scaler.fit_transform(
    pca_data
)


# ============================================================
# STEP 7 — PCA
# ============================================================

pca = PCA(
    n_components=10
)

pcs = pca.fit_transform(
    X
)


pca_df = pd.DataFrame(
    pcs,
    index=pca_data.index,
    columns=[
        f"PC{i}"
        for i in range(1, 11)
    ]
)

pca_df.index.name = "Patient ID"


print("\n======================================")
print("PCA VARIANCE EXPLAINED")
print("======================================")

for i, value in enumerate(
    pca.explained_variance_ratio_,
    start=1
):

    print(
        f"PC{i}: "
        f"{value * 100:.2f}%"
    )


# ============================================================
# STEP 8 — ROBUST PCA OUTLIER SCORE
# ============================================================
#
# We use first 5 PCs.
#
# For each PC:
#   robust z-score =
#   (value - median) / (1.4826 * MAD)
#
# Then combine PCs using Euclidean distance.
# ============================================================

robust_z = pd.DataFrame(
    index=pca_df.index
)


for pc in [
    "PC1",
    "PC2",
    "PC3",
    "PC4",
    "PC5"
]:

    values = pca_df[pc]

    median = values.median()

    mad = np.median(
        np.abs(
            values - median
        )
    )

    if mad == 0:

        robust_z[pc] = 0

    else:

        robust_z[pc] = (
            values - median
        ) / (
            1.4826 * mad
        )


pca_df["PCA_outlier_score"] = np.sqrt(
    (
        robust_z ** 2
    ).sum(axis=1)
)


# ============================================================
# STEP 9 — SAMPLE-TO-SAMPLE SPEARMAN CORRELATION
# ============================================================
#
# Rank-based correlation is useful because it is less sensitive
# to extreme TPM values.
# ============================================================

sample_corr = (
    qc_expression
    .corr(
        method="spearman"
    )
)


# Remove self-correlation = 1
mean_corr = []

median_corr = []

min_corr = []


for patient in sample_corr.columns:

    values = (
        sample_corr[patient]
        .drop(
            index=patient
        )
    )

    mean_corr.append(
        values.mean()
    )

    median_corr.append(
        values.median()
    )

    min_corr.append(
        values.min()
    )


correlation_qc = pd.DataFrame({

    "Patient ID":
        sample_corr.columns,

    "Mean Spearman to others":
        mean_corr,

    "Median Spearman to others":
        median_corr,

    "Minimum Spearman to another sample":
        min_corr
})


# ============================================================
# STEP 10 — COMBINE QC INFORMATION
# ============================================================

pca_out = (
    pca_df
    .reset_index()
)

combined = (
    sample_qc
    .merge(
        pca_out,
        on="Patient ID",
        how="inner"
    )
    .merge(
        correlation_qc,
        on="Patient ID",
        how="inner"
    )
)


# ============================================================
# STEP 11 — ROBUST QC RANKING
# ============================================================

combined["Detection_rank"] = (
    combined[
        "Detected genes TPM>=1"
    ]
    .rank(
        ascending=True
    )
)


combined["Count_rank"] = (
    combined[
        "Total protein counts"
    ]
    .rank(
        ascending=True
    )
)


combined["Correlation_rank"] = (
    combined[
        "Median Spearman to others"
    ]
    .rank(
        ascending=True
    )
)


# ============================================================
# STEP 12 — FLAG ONLY FOR INSPECTION
# ============================================================
#
# IMPORTANT:
# Flags do NOT mean automatic exclusion.
#
# PCA score > 5 is only an inspection threshold.
# ============================================================

combined[
    "Flag_PCA"
] = (
    combined[
        "PCA_outlier_score"
    ] > 5
)


corr_threshold = (
    combined[
        "Median Spearman to others"
    ]
    .quantile(0.025)
)


combined[
    "Flag_low_correlation"
] = (

    combined[
        "Median Spearman to others"
    ]
    <= corr_threshold
)


detection_threshold = (
    combined[
        "Detected genes TPM>=1"
    ]
    .quantile(0.025)
)


combined[
    "Flag_low_detection"
] = (

    combined[
        "Detected genes TPM>=1"
    ]
    <= detection_threshold
)


combined[
    "Number_QC_flags"
] = (

    combined[
        [
            "Flag_PCA",
            "Flag_low_correlation",
            "Flag_low_detection"
        ]
    ]
    .sum(axis=1)
)


# ============================================================
# STEP 13 — PRINT QC RESULTS
# ============================================================

print("\n======================================")
print("SAMPLE CORRELATION SUMMARY")
print("======================================")

print(

    combined[
        [
            "Mean Spearman to others",
            "Median Spearman to others"
        ]
    ]
    .describe()
    .to_string()

)


print("\n======================================")
print("PCA OUTLIER SCORE SUMMARY")
print("======================================")

print(

    combined[
        "PCA_outlier_score"
    ]
    .describe()
    .to_string()

)


print("\n======================================")
print("MOST EXTREME PCA SAMPLES")
print("======================================")

print(

    combined[
        [
            "Patient ID",
            "PCA_outlier_score",
            "Median Spearman to others",
            "Detected genes TPM>=1",
            "Total protein counts",
            "Number_QC_flags"
        ]
    ]
    .sort_values(
        "PCA_outlier_score",
        ascending=False
    )
    .head(15)
    .to_string(
        index=False
    )

)


print("\n======================================")
print("LOWEST CORRELATION SAMPLES")
print("======================================")

print(

    combined[
        [
            "Patient ID",
            "Median Spearman to others",
            "PCA_outlier_score",
            "Detected genes TPM>=1",
            "Total protein counts",
            "Number_QC_flags"
        ]
    ]
    .sort_values(
        "Median Spearman to others"
    )
    .head(15)
    .to_string(
        index=False
    )

)


print("\n======================================")
print("QC FLAG SUMMARY")
print("======================================")

print(
    combined[
        "Number_QC_flags"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print("\nPatients with >=2 QC flags:")

print(

    combined[
        combined[
            "Number_QC_flags"
        ] >= 2
    ][
        [
            "Patient ID",
            "PCA_outlier_score",
            "Median Spearman to others",
            "Detected genes TPM>=1",
            "Total protein counts",
            "Number_QC_flags"
        ]
    ]
    .sort_values(
        "Number_QC_flags",
        ascending=False
    )
    .to_string(
        index=False
    )

)


# ============================================================
# STEP 14 — PCA PLOT
# ============================================================

plt.figure(
    figsize=(9, 7)
)

plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"]
)

plt.xlabel(
    f"PC1 "
    f"({pca.explained_variance_ratio_[0]*100:.1f}%)"
)

plt.ylabel(
    f"PC2 "
    f"({pca.explained_variance_ratio_[1]*100:.1f}%)"
)

plt.title(
    "TCGA-GBM IDH-WT RNA-seq QC PCA"
)


# Label 10 most extreme samples
extreme = (
    pca_df
    .sort_values(
        "PCA_outlier_score",
        ascending=False
    )
    .head(10)
)


for patient, row in extreme.iterrows():

    plt.annotate(
        patient,
        (
            row["PC1"],
            row["PC2"]
        ),
        fontsize=7
    )


plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "TCGA_GBM_PCA_QC.png",
    dpi=300
)

plt.show()


# ============================================================
# STEP 15 — SAVE RESULTS
# ============================================================

combined.to_csv(

    OUTPUT_DIR /
    "TCGA_GBM_144_expression_QC_complete.csv",

    index=False

)


pca_df.to_csv(

    OUTPUT_DIR /
    "TCGA_GBM_144_PCA_coordinates.csv"

)


sample_corr.to_csv(

    OUTPUT_DIR /
    "TCGA_GBM_144_sample_spearman_matrix.csv"

)


pd.DataFrame({

    "ensembl_id":
        top_genes,

    "Variance_log2TPM":
        gene_variance.loc[
            top_genes
        ].values

}).to_csv(

    OUTPUT_DIR /
    "TCGA_GBM_QC_top5000_variable_genes.csv",

    index=False

)


print("\n======================================")
print("FILES SAVED")
print("======================================")

print(OUTPUT_DIR)

### STEP 1 — PATHS

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# STEP 1 — PATHS
# ============================================================

DOWNLOADS = BASE

GENE_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE2"
    / "TCGA_GBM_gene_annotation.csv"
)

# Search STRING files automatically
link_files = list(
    DOWNLOADS.glob(
        "9606.protein.links.v12.0.min700.onlyAB.txt.gz"
    )
)

info_files = list(
    DOWNLOADS.glob(
        "9606.protein.info.v12.0.txt.gz"
    )
)

if len(link_files) == 0:
    raise FileNotFoundError(
        "STRING protein.links file not found in Downloads."
    )

if len(info_files) == 0:
    raise FileNotFoundError(
        "STRING protein.info file not found in Downloads."
    )

STRING_LINKS = link_files[0]
STRING_INFO = info_files[0]

print("STRING links:")
print(STRING_LINKS)

print("\nSTRING protein info:")
print(STRING_INFO)

print("\nTCGA gene annotation:")
print(GENE_FILE)


# ============================================================
# STEP 2 — LOAD OUR 19,944 GENES
# ============================================================

genes = pd.read_csv(
    GENE_FILE,
    low_memory=False
)

print("\n======================================")
print("TCGA GENES")
print("======================================")

print("Rows:", len(genes))
print("Unique Ensembl genes:", genes["ensembl_id"].nunique())
print("Unique gene names:", genes["gene_name"].nunique())

print("\nColumns:")
print(genes.columns.tolist())


# ============================================================
# STEP 3 — LOAD STRING PROTEIN INFORMATION
# ============================================================

string_info = pd.read_csv(
    STRING_INFO,
    sep="\t",
    compression="gzip",
    low_memory=False
)

print("\n======================================")
print("STRING PROTEIN INFO")
print("======================================")

print("Rows:", len(string_info))

print("\nColumns:")
print(string_info.columns.tolist())

print("\nFirst 5 rows:")
print(string_info.head().to_string(index=False))


# ============================================================
# STEP 4 — STANDARDIZE STRING ID COLUMN
# ============================================================

# STRING normally uses:
# #string_protein_id

if "#string_protein_id" in string_info.columns:

    string_info = string_info.rename(
        columns={
            "#string_protein_id":
            "string_protein_id"
        }
    )

elif "string_protein_id" not in string_info.columns:

    raise ValueError(
        "Cannot identify STRING protein ID column."
    )


# ============================================================
# STEP 5 — CLEAN GENE SYMBOLS
# ============================================================

genes["gene_name"] = (
    genes["gene_name"]
    .astype(str)
    .str.strip()
)

string_info["preferred_name"] = (
    string_info["preferred_name"]
    .astype(str)
    .str.strip()
)


# ============================================================
# STEP 6 — INTERSECT TCGA GENES WITH STRING
# ============================================================

mapped = genes.merge(
    string_info[
        [
            "string_protein_id",
            "preferred_name"
        ]
    ],
    left_on="gene_name",
    right_on="preferred_name",
    how="inner"
)

print("\n======================================")
print("TCGA ↔ STRING MAPPING")
print("======================================")

print(
    "TCGA protein-coding genes:",
    genes["ensembl_id"].nunique()
)

print(
    "Genes mapped to STRING:",
    mapped["ensembl_id"].nunique()
)

print(
    "STRING proteins obtained:",
    mapped["string_protein_id"].nunique()
)

mapping_percent = (
    100
    * mapped["ensembl_id"].nunique()
    / genes["ensembl_id"].nunique()
)

print(
    "Mapping percentage:",
    round(mapping_percent, 2),
    "%"
)


# ============================================================
# STEP 7 — CHECK AMBIGUOUS MAPPINGS
# ============================================================

string_per_gene = (
    mapped
    .groupby("ensembl_id")["string_protein_id"]
    .nunique()
)

ambiguous_genes = string_per_gene[
    string_per_gene > 1
]

print("\n======================================")
print("MAPPING AMBIGUITY")
print("======================================")

print(
    "Genes mapped to >1 STRING protein:",
    len(ambiguous_genes)
)


# ============================================================
# STEP 8 — LOAD HIGH-CONFIDENCE STRING LINKS
# ============================================================

print("\nLoading STRING high-confidence network...")

links = pd.read_csv(
    STRING_LINKS,
    sep=r"\s+",
    compression="gzip",
    low_memory=False
)

print("\n======================================")
print("STRING NETWORK")
print("======================================")

print("Interaction rows:", len(links))

print("\nColumns:")
print(links.columns.tolist())

print("\nFirst 5 interactions:")
print(links.head().to_string(index=False))


# ============================================================
# STEP 9 — IDENTIFY LINK COLUMNS
# ============================================================

expected = {
    "protein1",
    "protein2",
    "combined_score"
}

missing = expected - set(links.columns)

if missing:
    raise ValueError(
        f"Missing STRING columns: {missing}"
    )


# ============================================================
# STEP 10 — KEEP ONLY OUR TCGA-MAPPED PROTEINS
# ============================================================

our_string_ids = set(
    mapped["string_protein_id"]
    .dropna()
    .unique()
)

tcga_network = links[
    links["protein1"].isin(our_string_ids)
    &
    links["protein2"].isin(our_string_ids)
].copy()


print("\n======================================")
print("TCGA STRING BACKBONE")
print("======================================")

print(
    "Edges retained:",
    len(tcga_network)
)

network_nodes = set(
    tcga_network["protein1"]
).union(
    set(tcga_network["protein2"])
)

print(
    "STRING proteins in network:",
    len(network_nodes)
)


# ============================================================
# STEP 11 — ADD GENE NAMES
# ============================================================

id_to_name = (
    string_info[
        [
            "string_protein_id",
            "preferred_name"
        ]
    ]
    .drop_duplicates(
        "string_protein_id"
    )
    .set_index(
        "string_protein_id"
    )["preferred_name"]
    .to_dict()
)

tcga_network["gene1"] = (
    tcga_network["protein1"]
    .map(id_to_name)
)

tcga_network["gene2"] = (
    tcga_network["protein2"]
    .map(id_to_name)
)


# ============================================================
# STEP 12 — SAVE SMALL NETWORK FILES
# ============================================================

OUT_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
)

OUT_DIR.mkdir(
    exist_ok=True
)

mapped.to_csv(
    OUT_DIR
    / "TCGA_GBM_gene_to_STRING_mapping.csv",
    index=False
)

tcga_network.to_csv(
    OUT_DIR
    / "TCGA_GBM_STRING_high_confidence_backbone.csv",
    index=False
)


# ============================================================
# STEP 13 — FINAL SUMMARY
# ============================================================

summary = pd.DataFrame({

    "Metric": [
        "TCGA protein-coding genes",
        "Genes mapped to STRING",
        "STRING proteins mapped",
        "STRING network nodes",
        "STRING high-confidence edges"
    ],

    "N": [
        genes["ensembl_id"].nunique(),
        mapped["ensembl_id"].nunique(),
        mapped["string_protein_id"].nunique(),
        len(network_nodes),
        len(tcga_network)
    ]

})

print("\n======================================")
print("FINAL SUMMARY")
print("======================================")

print(
    summary.to_string(index=False)
)

summary.to_csv(
    OUT_DIR
    / "TCGA_GBM_STRING_backbone_summary.csv",
    index=False
)

print("\nFiles saved in:")
print(OUT_DIR)

### NEXT STEP

In [ ]:
# ============================================================
# NEXT STEP
# Align TCGA expression with the STRING network backbone
#
# INPUT:
#   19,944 genes x 144 patients
#   STRING high-confidence network
#
# OUTPUT:
#   STRING-node expression matrix x 144 patients
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. PATHS
# ============================================================

DOWNLOADS = BASE

TPM_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE2"
    / "TCGA_GBM_144_protein_coding_TPM_matrix.csv"
)

MAPPING_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
    / "TCGA_GBM_gene_to_STRING_mapping.csv"
)

NETWORK_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
    / "TCGA_GBM_STRING_high_confidence_backbone.csv"
)

OUTPUT_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
    / "PATIENT_NETWORK_INPUT"
)

OUTPUT_DIR.mkdir(exist_ok=True)


# ============================================================
# 2. LOAD DATA
# ============================================================

tpm = pd.read_csv(
    TPM_FILE,
    index_col=0
)

mapping = pd.read_csv(
    MAPPING_FILE,
    low_memory=False
)

network = pd.read_csv(
    NETWORK_FILE,
    low_memory=False
)


print("\n======================================")
print("INPUT")
print("======================================")

print("TPM genes:", tpm.shape[0])
print("Patients:", tpm.shape[1])

print(
    "Mapped Ensembl genes:",
    mapping["ensembl_id"].nunique()
)

print(
    "Mapped STRING proteins:",
    mapping["string_protein_id"].nunique()
)

print(
    "STRING network edges:",
    len(network)
)


# ============================================================
# 3. GET EXACT STRING NODES USED IN OUR NETWORK
# ============================================================

network_nodes = set(
    network["protein1"]
).union(
    set(network["protein2"])
)


print("\n======================================")
print("NETWORK NODES")
print("======================================")

print(
    "STRING proteins in backbone:",
    len(network_nodes)
)


# ============================================================
# 4. KEEP ONLY MAPPINGS FOR ACTUAL NETWORK NODES
# ============================================================

network_mapping = mapping[
    mapping["string_protein_id"]
    .isin(network_nodes)
].copy()


print("\n======================================")
print("NETWORK MAPPING")
print("======================================")

print(
    "Ensembl genes represented:",
    network_mapping["ensembl_id"].nunique()
)

print(
    "STRING proteins represented:",
    network_mapping["string_protein_id"].nunique()
)


# ============================================================
# 5. CHECK MANY-TO-ONE MAPPING
#
# Sometimes two Ensembl records may map to the same STRING
# protein. We must not accidentally create duplicate nodes.
# ============================================================

mapping_counts = (
    network_mapping
    .groupby("string_protein_id")["ensembl_id"]
    .nunique()
)

duplicated_string_nodes = mapping_counts[
    mapping_counts > 1
]


print("\n======================================")
print("MAPPING DUPLICATES")
print("======================================")

print(
    "STRING nodes linked to >1 Ensembl gene:",
    len(duplicated_string_nodes)
)


if len(duplicated_string_nodes) > 0:

    print("\nExamples:")

    print(
        network_mapping[
            network_mapping["string_protein_id"]
            .isin(duplicated_string_nodes.index)
        ][
            [
                "ensembl_id",
                "gene_name",
                "string_protein_id",
                "preferred_name"
            ]
        ]
        .sort_values("string_protein_id")
        .head(30)
        .to_string(index=False)
    )


# ============================================================
# 6. CALCULATE MEDIAN EXPRESSION OF EACH ENSEMBL GENE
#
# Used ONLY if several Ensembl IDs map to the same STRING node.
#
# We choose the most informative expressed record.
# This does NOT use survival.
# ============================================================

gene_medians = (
    tpm.median(axis=1)
    .rename("Median_TPM")
)


network_mapping = network_mapping.merge(
    gene_medians,
    left_on="ensembl_id",
    right_index=True,
    how="left"
)


# ============================================================
# 7. ONE ENSEMBL GENE PER STRING NODE
#
# If duplicated:
# choose highest median TPM.
# Final tie-break = Ensembl ID.
# ============================================================

resolved_mapping = (
    network_mapping
    .sort_values(
        [
            "string_protein_id",
            "Median_TPM",
            "ensembl_id"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .drop_duplicates(
        "string_protein_id",
        keep="first"
    )
    .copy()
)


print("\n======================================")
print("RESOLVED MAPPING")
print("======================================")

print(
    "Final STRING nodes:",
    resolved_mapping["string_protein_id"].nunique()
)

print(
    "Final unique Ensembl genes:",
    resolved_mapping["ensembl_id"].nunique()
)


# ============================================================
# 8. VERIFY THAT EVERY SELECTED ENSEMBL ID EXISTS IN TPM
# ============================================================

missing_expression = (
    set(resolved_mapping["ensembl_id"])
    -
    set(tpm.index)
)


print("\nMissing expression genes:")
print(len(missing_expression))


if len(missing_expression) > 0:

    print(
        list(missing_expression)[:20]
    )


# ============================================================
# 9. BUILD STRING-NODE TPM MATRIX
# ============================================================

resolved_mapping = resolved_mapping[
    resolved_mapping["ensembl_id"]
    .isin(tpm.index)
].copy()


selected_ensembl = (
    resolved_mapping["ensembl_id"]
    .tolist()
)


string_expression = (
    tpm
    .loc[selected_ensembl]
    .copy()
)


# Replace Ensembl row names with STRING protein IDs
ensembl_to_string = (
    resolved_mapping
    .set_index("ensembl_id")["string_protein_id"]
    .to_dict()
)


string_expression.index = (
    string_expression.index
    .map(ensembl_to_string)
)

string_expression.index.name = (
    "string_protein_id"
)


# ============================================================
# 10. CREATE log2(TPM + 1)
#
# This will later be more suitable for network estimation.
# We keep raw TPM too.
# ============================================================

log_string_expression = np.log2(
    string_expression + 1
)


# ============================================================
# 11. VALIDATION
# ============================================================

print("\n======================================")
print("FINAL STRING EXPRESSION MATRIX")
print("======================================")

print(
    "Nodes:",
    string_expression.shape[0]
)

print(
    "Patients:",
    string_expression.shape[1]
)

print(
    "Unique nodes:",
    string_expression.index.nunique()
)

print(
    "Unique patients:",
    string_expression.columns.nunique()
)

print(
    "Missing TPM values:",
    int(
        string_expression
        .isna()
        .sum()
        .sum()
    )
)


# ============================================================
# 12. CHECK NETWORK COVERAGE
# ============================================================

expression_nodes = set(
    string_expression.index
)

missing_network_nodes = (
    network_nodes
    -
    expression_nodes
)


print("\n======================================")
print("NETWORK COVERAGE")
print("======================================")

print(
    "Network nodes:",
    len(network_nodes)
)

print(
    "Nodes with expression:",
    len(expression_nodes)
)

print(
    "Network nodes without expression:",
    len(missing_network_nodes)
)

coverage = (
    100
    * len(expression_nodes)
    / len(network_nodes)
)

print(
    "Coverage:",
    round(coverage, 2),
    "%"
)


# ============================================================
# 13. ADD GENE NAMES TO NODE TABLE
# ============================================================

node_table = resolved_mapping[
    [
        "string_protein_id",
        "ensembl_id",
        "gene_name",
        "preferred_name",
        "Median_TPM"
    ]
].copy()


# ============================================================
# 14. SAVE
# ============================================================

string_expression.to_csv(
    OUTPUT_DIR
    / "TCGA_GBM_144_STRING_TPM_matrix.csv"
)

log_string_expression.to_csv(
    OUTPUT_DIR
    / "TCGA_GBM_144_STRING_log2TPM_matrix.csv"
)

node_table.to_csv(
    OUTPUT_DIR
    / "TCGA_GBM_STRING_node_annotation.csv",
    index=False
)

resolved_mapping.to_csv(
    OUTPUT_DIR
    / "TCGA_GBM_STRING_mapping_resolved.csv",
    index=False
)


# ============================================================
# 15. SUMMARY
# ============================================================

summary = pd.DataFrame({

    "Metric": [
        "Original TCGA genes",
        "STRING network nodes",
        "STRING nodes with RNA",
        "Patients",
        "Missing expression values"
    ],

    "N": [
        tpm.shape[0],
        len(network_nodes),
        string_expression.shape[0],
        string_expression.shape[1],
        int(
            string_expression
            .isna()
            .sum()
            .sum()
        )
    ]

})


print("\n======================================")
print("FINAL SUMMARY")
print("======================================")

print(
    summary.to_string(index=False)
)


summary.to_csv(
    OUTPUT_DIR
    / "STRING_expression_alignment_summary.csv",
    index=False
)


print("\nFiles saved in:")
print(OUTPUT_DIR)

### RECOUNT3 DIAGNOSTIC

In [ ]:
"""
RECOUNT3 DIAGNOSTIC
===================

Goal:
1. Check whether recount3 contains our 144 TCGA-GBM patients.
2. Identify the CORRECT TCGA identifier column automatically.
3. Check GTEx BRAIN availability.
4. Do NOT download expression matrices yet.

This script only downloads metadata.
"""

import gzip
import io
from pathlib import Path

import pandas as pd
import requests


# ============================================================
# CONFIGURATION
# ============================================================

DOWNLOADS = BASE

PATIENT_FILE = (
    DOWNLOADS /
    "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.tsv"
)

PATIENT_ID_COLUMN = "Patient ID"

# recount3 mirror
BASE = "https://recount-opendata.s3.amazonaws.com/recount3/release/human/data_sources"

TCGA_PROJECT = "GBM"
GTEX_PROJECT = "BRAIN"

OUTPUT_DIR = DOWNLOADS / "RECOUNT3_DIAGNOSTIC"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# FUNCTIONS
# ============================================================

def section(title):
    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)


def fetch_gz_tsv(url):
    """
    Download a small recount3 metadata .gz file
    and return it as a pandas DataFrame.
    """
    print("Downloading:")
    print(url)

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with gzip.open(io.BytesIO(response.content), "rt") as f:
        return pd.read_csv(
            f,
            sep="\t",
            low_memory=False
        )


def normalize_tcga_patient(value):
    """
    Convert a TCGA sample barcode to patient ID.

    Example:
    TCGA-06-0125-01A-01R...
          ->
    TCGA-06-0125
    """

    if pd.isna(value):
        return None

    value = str(value).strip()

    # remove recount3 suffix such as ".1"
    value = value.split(".")[0]

    if not value.startswith("TCGA-"):
        return None

    parts = value.split("-")

    if len(parts) < 3:
        return None

    return "-".join(parts[:3])


# ============================================================
# 0. LOAD OUR 144 PATIENTS
# ============================================================

section("0 — OUR TCGA-GBM COHORT")

cohort = pd.read_csv(
    PATIENT_FILE,
    sep="\t",
    low_memory=False
)

if PATIENT_ID_COLUMN not in cohort.columns:
    raise ValueError(
        f"Column '{PATIENT_ID_COLUMN}' not found.\n"
        f"Available columns:\n{cohort.columns.tolist()}"
    )

our_patients = sorted(
    cohort[PATIENT_ID_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

our_patient_set = set(our_patients)

print("Patients in our cohort:", len(our_patients))

print("\nFirst 10:")
for p in our_patients[:10]:
    print(p)


# ============================================================
# 1. DOWNLOAD TCGA-GBM METADATA
# ============================================================

section("1 — RECOUNT3 TCGA-GBM METADATA")

# GBM -> last two characters = BM
tcga_url = (
    f"{BASE}/tcga/metadata/BM/GBM/"
    f"tcga.tcga.GBM.MD.gz"
)

try:

    tcga = fetch_gz_tsv(tcga_url)

except Exception as e:

    print("\nTCGA metadata download failed:")
    print(e)

    raise


print("\nTCGA-GBM metadata rows:", len(tcga))
print("Metadata columns:", len(tcga.columns))

print("\nCOLUMN NAMES:")

for c in tcga.columns:
    print(c)


print("\nFIRST 5 ROWS:")
print(
    tcga.head()
    .to_string()
)


# ============================================================
# 2. FIND THE CORRECT TCGA ID COLUMN
# ============================================================

section("2 — IDENTIFY TCGA BARCODE COLUMN")

candidate_columns = []

for col in tcga.columns:

    col_lower = str(col).lower()

    if (
        "id" in col_lower
        or
        "barcode" in col_lower
        or
        "sample" in col_lower
    ):
        candidate_columns.append(col)


print("Candidate columns:")

for c in candidate_columns:
    print(" -", c)


results = []

for col in candidate_columns:

    normalized = (
        tcga[col]
        .apply(normalize_tcga_patient)
        .dropna()
    )

    recount_patients = set(normalized)

    matched = our_patient_set & recount_patients

    results.append({
        "Column": col,
        "Valid TCGA patient IDs": len(recount_patients),
        "Our patients matched": len(matched)
    })


column_test = pd.DataFrame(results)

column_test = column_test.sort_values(
    "Our patients matched",
    ascending=False
)


print("\nMATCHING PERFORMANCE BY COLUMN:")

print(
    column_test
    .to_string(index=False)
)


if len(column_test) == 0:

    raise RuntimeError(
        "No possible identifier column found."
    )


best_column = column_test.iloc[0]["Column"]
best_matches = int(
    column_test.iloc[0]["Our patients matched"]
)


print("\nBEST IDENTIFIER COLUMN:")
print(best_column)

print("\nOur patients matched using this column:")
print(best_matches)


# ============================================================
# 3. FINAL 144-PATIENT MATCH
# ============================================================

section("3 — OUR 144 PATIENTS vs RECOUNT3")

tcga["Normalized Patient ID"] = (
    tcga[best_column]
    .apply(normalize_tcga_patient)
)

recount_patient_set = set(
    tcga["Normalized Patient ID"]
    .dropna()
)


matched = sorted(
    our_patient_set &
    recount_patient_set
)

missing = sorted(
    our_patient_set -
    recount_patient_set
)


print("Our original patients:", len(our_patients))

print(
    "Patients found in recount3:",
    len(matched)
)

print(
    "Patients missing from recount3:",
    len(missing)
)


print("\nMATCH PERCENTAGE:")

print(
    round(
        100 * len(matched) / len(our_patients),
        2
    ),
    "%"
)


if missing:

    print("\nMISSING PATIENTS:")

    for p in missing:
        print(p)


# ============================================================
# 4. CHECK HOW MANY RECOUNT3 SAMPLES EACH PATIENT HAS
# ============================================================

section("4 — SAMPLE COUNTS PER MATCHED PATIENT")

sample_counts = (
    tcga[
        tcga["Normalized Patient ID"]
        .isin(our_patient_set)
    ]
    ["Normalized Patient ID"]
    .value_counts()
)


print(sample_counts.value_counts().sort_index())

multiple = sample_counts[
    sample_counts > 1
]


print(
    "\nPatients with >1 recount3 sample:",
    len(multiple)
)


if len(multiple) > 0:

    print("\nExamples:")

    print(
        multiple.head(20)
        .to_string()
    )


# ============================================================
# 5. DOWNLOAD GTEX BRAIN METADATA
# ============================================================

section("5 — RECOUNT3 GTEx BRAIN")

# BRAIN -> last two characters = IN
gtex_url = (
    f"{BASE}/gtex/metadata/IN/BRAIN/"
    f"gtex.gtex.BRAIN.MD.gz"
)

try:

    gtex = fetch_gz_tsv(gtex_url)

except Exception as e:

    print("\nGTEx BRAIN metadata download failed:")
    print(e)

    raise


print("\nGTEx BRAIN metadata rows:", len(gtex))
print("Metadata columns:", len(gtex.columns))


print("\nFIRST 5 ROWS:")

print(
    gtex.head()
    .to_string()
)


# ============================================================
# 6. GTEX BRAIN REGIONS
# ============================================================

section("6 — GTEx BRAIN REGIONS")

possible_region_columns = [
    "gtex.smtsd",
    "smtsd",
    "gtex.smts",
    "smts",
    "tissue",
    "primary_site"
]


region_column = None

for c in possible_region_columns:

    if c in gtex.columns:

        region_column = c
        break


if region_column:

    print(
        "Brain region column:",
        region_column
    )

    print("\nSamples per brain region:")

    print(
        gtex[region_column]
        .value_counts(dropna=False)
        .to_string()
    )

else:

    print(
        "No brain-region column was automatically identified."
    )

    print(
        "\nColumns containing 'tissue', 'site', 'smts' or 'brain':"
    )

    for c in gtex.columns:

        x = c.lower()

        if (
            "tissue" in x
            or
            "site" in x
            or
            "smts" in x
            or
            "brain" in x
        ):
            print(c)


# ============================================================
# 7. SAVE RESULTS
# ============================================================

section("7 — SAVE DIAGNOSTIC RESULTS")


patient_check = pd.DataFrame({
    "Patient ID": our_patients
})

patient_check["Found in recount3"] = (
    patient_check["Patient ID"]
    .isin(recount_patient_set)
)


# number of recount3 records for each patient
count_dictionary = (
    tcga[
        tcga["Normalized Patient ID"]
        .isin(our_patient_set)
    ]
    ["Normalized Patient ID"]
    .value_counts()
    .to_dict()
)


patient_check["N recount3 samples"] = (
    patient_check["Patient ID"]
    .map(count_dictionary)
    .fillna(0)
    .astype(int)
)


patient_check.to_csv(
    OUTPUT_DIR /
    "TCGA_GBM_144_recount3_patient_check.csv",
    index=False
)


column_test.to_csv(
    OUTPUT_DIR /
    "TCGA_GBM_recount3_identifier_column_test.csv",
    index=False
)


gtex.to_csv(
    OUTPUT_DIR /
    "GTEx_BRAIN_recount3_metadata.csv",
    index=False
)


# ============================================================
# 8. FINAL DECISION SUMMARY
# ============================================================

section("FINAL RESULT")

print("Our TCGA-GBM cohort:", len(our_patients))

print(
    "Found in recount3:",
    len(matched)
)

print(
    "Missing:",
    len(missing)
)

print(
    "GTEx BRAIN samples available:",
    len(gtex)
)


print("\nInterpretation:")

if len(matched) == len(our_patients):

    print(
        "SUCCESS: all 144 patients are represented in recount3."
    )

    print(
        "We can proceed to expression extraction."
    )

elif len(matched) >= 130:

    print(
        "Very good coverage, but not complete."
    )

    print(
        "We must inspect the few missing patients before proceeding."
    )

elif len(matched) > 94:

    print(
        "recount3 improves substantially over Toil,"
        " but does not recover the complete cohort."
    )

    print(
        "We should inspect the missing cases before deciding."
    )

else:

    print(
        "recount3 does not solve our 94-patient problem."
    )

    print(
        "Do NOT download the large expression matrices yet."
    )


print("\nResults saved in:")
print(OUTPUT_DIR)

print("\nDONE.")

### CONFIGURATION

In [ ]:
import gzip
import io
import re
from pathlib import Path

import pandas as pd
import requests


# ============================================================
# CONFIGURATION
# ============================================================

DOWNLOADS = BASE

PATIENT_FILE = (
    DOWNLOADS /
    "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.tsv"
)

PATIENT_ID_COLUMN = "Patient ID"

OUTDIR = DOWNLOADS / "TCGA_GBM_STAGE_A_DISCOVERY"
OUTDIR.mkdir(parents=True, exist_ok=True)

BASE = "http://duffel.rail.bio/recount3/human/data_sources"

TCGA_PROJECT = "GBM"
GTEX_PROJECT = "BRAIN"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def section(title):
    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)


def download_gz_table(url):
    print("Downloading metadata:")
    print(url)

    r = requests.get(url, timeout=180)
    r.raise_for_status()

    with gzip.open(io.BytesIO(r.content), "rt") as f:
        return pd.read_csv(
            f,
            sep="\t",
            low_memory=False
        )


def project_shard(project):
    return project[-2:]


def clean_string(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def tcga_patient_id(x):
    """
    Extract TCGA patient ID:
    TCGA-XX-XXXX
    """

    x = clean_string(x)

    m = re.search(
        r"(TCGA-[A-Za-z0-9]{2}-[A-Za-z0-9]{4})",
        x,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1).upper()

    return None


def gtex_donor_id(x):
    """
    Example:
    GTEX-1117F-0226-SM-5GZZ7
    ->
    GTEX-1117F
    """

    x = clean_string(x).upper()

    m = re.search(
        r"(GTEX-[A-Z0-9]+)",
        x
    )

    if m:
        return m.group(1)

    return None


def find_best_id_column(df, source):

    candidates = []

    preferred = [
        "external_id",
        "sample_id",
        "sample",
        "submitter_id",
        "barcode"
    ]

    for p in preferred:
        for c in df.columns:
            if c.lower() == p.lower():
                candidates.append(c)

    for c in df.columns:
        lc = c.lower()

        if (
            "external" in lc
            or "sample" in lc
            or "barcode" in lc
        ):
            if c not in candidates:
                candidates.append(c)

    if not candidates:
        raise ValueError(
            f"No possible sample-ID column found for {source}"
        )

    # test which column actually contains TCGA / GTEX identifiers

    best = None
    best_n = -1

    prefix = "TCGA-" if source == "TCGA" else "GTEX-"

    for c in candidates:

        n = (
            df[c]
            .astype(str)
            .str.upper()
            .str.contains(prefix, regex=False)
            .sum()
        )

        if n > best_n:
            best_n = n
            best = c

    print(
        f"Selected ID column for {source}:",
        best,
        f"({best_n} recognizable IDs)"
    )

    return best


# ============================================================
# 1. LOAD OUR 144 SURVIVAL PATIENTS
# ============================================================

section("1 -- OUR SURVIVAL COHORT")

our_df = pd.read_csv(
    PATIENT_FILE,
    sep="\t",
    low_memory=False
)

our_patients = (
    our_df[PATIENT_ID_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
)

our_patients = set(our_patients)

print("Patients in survival cohort:", len(our_patients))

if len(our_patients) != 144:
    print(
        "WARNING: expected 144 patients but found",
        len(our_patients)
    )


# ============================================================
# 2. DOWNLOAD TCGA-GBM METADATA
# ============================================================

section("2 -- RECOUNT3 TCGA-GBM")

tcga_shard = project_shard(TCGA_PROJECT)

tcga_url = (
    f"{BASE}/tcga/metadata/"
    f"{tcga_shard}/{TCGA_PROJECT}/"
    f"tcga.tcga.{TCGA_PROJECT}.MD.gz"
)

tcga = download_gz_table(tcga_url)

print("Metadata rows:", len(tcga))
print("Columns:", len(tcga.columns))

tcga_id_col = find_best_id_column(
    tcga,
    "TCGA"
)

tcga["Recount sample ID"] = (
    tcga[tcga_id_col]
    .astype(str)
    .str.strip()
)

tcga["Patient ID"] = (
    tcga["Recount sample ID"]
    .apply(tcga_patient_id)
)

tcga = tcga[
    tcga["Patient ID"].notna()
].copy()

print(
    "Rows with recognizable TCGA patient ID:",
    len(tcga)
)

print(
    "Unique TCGA-GBM patients:",
    tcga["Patient ID"].nunique()
)


# ============================================================
# 3. IDENTIFY PRIMARY TUMORS
# ============================================================

section("3 -- TCGA PRIMARY-TUMOR SELECTION")

# TCGA barcode sample type:
# -01 = Primary Solid Tumor
#
# We extract the two digits immediately following patient ID.

def tcga_sample_type_code(sample):

    sample = clean_string(sample).upper()

    m = re.search(
        r"TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-(\d{2})",
        sample
    )

    if m:
        return m.group(1)

    return None


tcga["TCGA sample type code"] = (
    tcga["Recount sample ID"]
    .apply(tcga_sample_type_code)
)

print("Sample type codes:")

print(
    tcga["TCGA sample type code"]
    .value_counts(dropna=False)
    .head(20)
)


tcga_primary = tcga[
    tcga["TCGA sample type code"] == "01"
].copy()

print(
    "\nPrimary tumor samples:",
    len(tcga_primary)
)

print(
    "Primary tumor patients:",
    tcga_primary["Patient ID"].nunique()
)


# ============================================================
# 4. EXCLUDE OUR 144 SURVIVAL PATIENTS
# ============================================================

section("4 -- REMOVE SURVIVAL-COHORT PATIENTS")

tcga_primary["In our 144"] = (
    tcga_primary["Patient ID"]
    .isin(our_patients)
)

overlap = tcga_primary[
    tcga_primary["In our 144"]
].copy()

discovery_tumor = tcga_primary[
    ~tcga_primary["In our 144"]
].copy()

print(
    "Primary TCGA-GBM patients overlapping our 144:",
    overlap["Patient ID"].nunique()
)

print(
    "Primary TCGA-GBM patients remaining:",
    discovery_tumor["Patient ID"].nunique()
)


# ============================================================
# 5. ONE TUMOR SAMPLE PER PATIENT
# ============================================================

section("5 -- ONE TCGA SAMPLE PER PATIENT")

# deterministic selection
discovery_tumor = (
    discovery_tumor
    .sort_values(
        ["Patient ID", "Recount sample ID"]
    )
    .drop_duplicates(
        subset=["Patient ID"],
        keep="first"
    )
    .copy()
)

print(
    "Final independent GBM tumor samples:",
    len(discovery_tumor)
)

print(
    "Final independent GBM patients:",
    discovery_tumor["Patient ID"].nunique()
)


# ============================================================
# 6. DOWNLOAD GTEX-BRAIN METADATA
# ============================================================

section("6 -- RECOUNT3 GTEX-BRAIN")

gtex_shard = project_shard(GTEX_PROJECT)

gtex_url = (
    f"{BASE}/gtex/metadata/"
    f"{gtex_shard}/{GTEX_PROJECT}/"
    f"gtex.gtex.{GTEX_PROJECT}.MD.gz"
)

gtex = download_gz_table(gtex_url)

print("Metadata rows:", len(gtex))
print("Columns:", len(gtex.columns))

gtex_id_col = find_best_id_column(
    gtex,
    "GTEX"
)

gtex["Recount sample ID"] = (
    gtex[gtex_id_col]
    .astype(str)
    .str.strip()
)

gtex["GTEx donor ID"] = (
    gtex["Recount sample ID"]
    .apply(gtex_donor_id)
)

print(
    "Recognizable GTEx samples:",
    gtex["GTEx donor ID"].notna().sum()
)

print(
    "Unique GTEx donors:",
    gtex["GTEx donor ID"].nunique()
)


# ============================================================
# 7. SEARCH GTEX BRAIN REGION FIELDS
# ============================================================

section("7 -- GTEX BRAIN REGION METADATA")

region_candidates = []

keywords = [
    "smtsd",
    "smts",
    "tissue",
    "region",
    "site",
    "body"
]

for c in gtex.columns:

    lc = c.lower()

    if any(k in lc for k in keywords):
        region_candidates.append(c)


print("Candidate tissue/region columns:")

for c in region_candidates:

    print("\nCOLUMN:", c)

    values = (
        gtex[c]
        .dropna()
        .astype(str)
        .value_counts()
        .head(30)
    )

    print(values.to_string())


# ============================================================
# 8. FIND CORTICAL SAMPLES
# ============================================================

section("8 -- SELECT CORTICAL GTEX SAMPLES")

# We search across plausible metadata columns.
#
# Intended GTEx regions:
# Brain - Cortex
# Brain - Frontal Cortex (BA9)
#
# We deliberately do NOT use Cerebellum, spinal cord,
# substantia nigra, etc.

cortex_mask = pd.Series(
    False,
    index=gtex.index
)

matched_regions = []

for c in region_candidates:

    text = (
        gtex[c]
        .fillna("")
        .astype(str)
    )

    mask = (
        text.str.contains(
            "cortex",
            case=False,
            regex=False
        )
    )

    if mask.sum() > 0:

        cortex_mask = cortex_mask | mask

        values = (
            gtex.loc[mask, c]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        for v in values:
            matched_regions.append(
                (c, v)
            )


gtex_cortex = gtex[
    cortex_mask
].copy()


print("Matched cortical labels:")

for c, value in sorted(set(matched_regions)):
    print(f"{c}: {value}")


print(
    "\nGTEx cortical samples:",
    len(gtex_cortex)
)

print(
    "GTEx cortical donors:",
    gtex_cortex["GTEx donor ID"].nunique()
)


# ============================================================
# SAFETY CHECK
# ============================================================

if len(gtex_cortex) == 0:

    print("\n!!! IMPORTANT !!!")
    print(
        "No cortex samples were selected automatically."
    )

    print(
        "DO NOT continue to DE."
    )

    print(
        "Inspect the region labels printed above first."
    )

    raise RuntimeError(
        "Cortex selection returned zero samples."
    )


# ============================================================
# 9. REMOVE UNRECOGNIZED DONOR IDS
# ============================================================

gtex_cortex = gtex_cortex[
    gtex_cortex["GTEx donor ID"].notna()
].copy()


# ============================================================
# 10. ONE CORTEX SAMPLE PER GTEX DONOR
# ============================================================

section("9 -- ONE GTEX SAMPLE PER DONOR")

gtex_cortex = (
    gtex_cortex
    .sort_values(
        ["GTEx donor ID", "Recount sample ID"]
    )
)

# For now deterministic selection.
# We are NOT averaging multiple tissues from the same donor.

gtex_final = (
    gtex_cortex
    .drop_duplicates(
        subset=["GTEx donor ID"],
        keep="first"
    )
    .copy()
)

print(
    "Cortical samples before donor deduplication:",
    len(gtex_cortex)
)

print(
    "Unique cortical donors:",
    gtex_cortex["GTEx donor ID"].nunique()
)

print(
    "Final GTEx controls:",
    len(gtex_final)
)


# ============================================================
# 11. FINAL INDEPENDENCE CHECK
# ============================================================

section("10 -- FINAL COHORT CHECK")

tumor_patients = set(
    discovery_tumor["Patient ID"]
)

survival_overlap = (
    tumor_patients &
    our_patients
)

print(
    "Discovery GBM patients:",
    len(tumor_patients)
)

print(
    "GTEx normal donors:",
    gtex_final["GTEx donor ID"].nunique()
)

print(
    "Overlap between discovery tumors and our 144:",
    len(survival_overlap)
)


if len(survival_overlap) != 0:

    raise RuntimeError(
        "ERROR: discovery cohort still overlaps "
        "with survival cohort."
    )


# ============================================================
# 12. SAVE FINAL SAMPLE LISTS
# ============================================================

section("11 -- SAVE FILES")

tumor_output = (
    discovery_tumor[
        [
            "Recount sample ID",
            "Patient ID",
            "TCGA sample type code"
        ]
    ]
    .copy()
)

tumor_output["Group"] = "GBM"

normal_output = (
    gtex_final[
        [
            "Recount sample ID",
            "GTEx donor ID"
        ]
    ]
    .copy()
)

normal_output["Group"] = "Normal"


tumor_file = (
    OUTDIR /
    "STAGE_A_TCGA_GBM_INDEPENDENT_TUMORS.csv"
)

normal_file = (
    OUTDIR /
    "STAGE_A_GTEX_CORTEX_ONE_PER_DONOR.csv"
)

overlap_file = (
    OUTDIR /
    "STAGE_A_EXCLUDED_SURVIVAL_PATIENTS.csv"
)


tumor_output.to_csv(
    tumor_file,
    index=False
)

normal_output.to_csv(
    normal_file,
    index=False
)

overlap[
    [
        "Recount sample ID",
        "Patient ID"
    ]
].to_csv(
    overlap_file,
    index=False
)


# ============================================================
# 13. SUMMARY
# ============================================================

summary = pd.DataFrame(
    {
        "Metric": [
            "Our survival patients",
            "recount3 TCGA-GBM metadata rows",
            "TCGA primary tumors",
            "Overlap excluded",
            "Final independent GBM tumors",
            "GTEx-BRAIN metadata rows",
            "GTEx cortical samples",
            "Final unique GTEx cortical donors",
            "Discovery-survival patient overlap"
        ],
        "N": [
            len(our_patients),
            len(tcga),
            len(tcga_primary),
            overlap["Patient ID"].nunique(),
            len(discovery_tumor),
            len(gtex),
            len(gtex_cortex),
            len(gtex_final),
            len(survival_overlap)
        ]
    }
)

summary_file = (
    OUTDIR /
    "STAGE_A_SAMPLE_SELECTION_SUMMARY.csv"
)

summary.to_csv(
    summary_file,
    index=False
)


section("FINAL STAGE-A DESIGN")

print(summary.to_string(index=False))


print("\n======================================")
print("FILES CREATED")
print("======================================")

print(tumor_file)
print(normal_file)
print(overlap_file)
print(summary_file)


print("\n======================================")
print("NEXT STEP")
print("======================================")

print(
    "If the counts and cortical regions are correct, "
    "the next step is expression extraction + DE."
)

### GTEx CORTEX + ISCHEMIC TIME DIAGNOSTIC

In [ ]:
# ============================================================
# GTEx CORTEX + ISCHEMIC TIME DIAGNOSTIC
# ============================================================

# Primary cortical regions for this study
PRIMARY_CORTEX_REGIONS = [
    "Brain - Cortex",
    "Brain - Frontal Cortex (BA9)"
]

# Keep only these two regions explicitly
gtex_cortex_clean = gtex[
    gtex["SMTSD"].isin(PRIMARY_CORTEX_REGIONS)
].copy()

print("\n======================================")
print("PRIMARY CORTEX REGIONS")
print("======================================")

print(
    gtex_cortex_clean["SMTSD"]
    .value_counts()
    .to_string()
)

print("\nCortical samples:",
      len(gtex_cortex_clean))

print(
    "Unique donors:",
    gtex_cortex_clean["GTEx donor ID"].nunique()
)


# ============================================================
# ISCHEMIC TIME
# ============================================================

# recount3 metadata contains SMTSISCH
# sample ischemic time, generally in minutes

gtex_cortex_clean["SMTSISCH"] = pd.to_numeric(
    gtex_cortex_clean["SMTSISCH"],
    errors="coerce"
)

print("\n======================================")
print("ISCHEMIC TIME")
print("======================================")

print(
    gtex_cortex_clean["SMTSISCH"]
    .describe()
)

print(
    "\nMissing ischemic-time values:",
    gtex_cortex_clean["SMTSISCH"].isna().sum()
)

print(
    "Samples with ischemic time > 480 min:",
    (
        gtex_cortex_clean["SMTSISCH"] > 480
    ).sum()
)

print(
    "Samples with ischemic time <= 480 min:",
    (
        gtex_cortex_clean["SMTSISCH"] <= 480
    ).sum()
)


# ============================================================
# ONE SAMPLE PER DONOR
#
# Prefer:
# 1. sample with ischemic time available
# 2. lower ischemic time
# 3. deterministic sample ID as final tie-break
# ============================================================

gtex_cortex_clean["ischemia_missing"] = (
    gtex_cortex_clean["SMTSISCH"].isna()
)

gtex_cortex_clean = gtex_cortex_clean.sort_values(
    [
        "GTEx donor ID",
        "ischemia_missing",
        "SMTSISCH",
        "Recount sample ID"
    ],
    ascending=[
        True,
        True,
        True,
        True
    ]
)

gtex_final = (
    gtex_cortex_clean
    .drop_duplicates(
        subset="GTEx donor ID",
        keep="first"
    )
    .copy()
)

print("\n======================================")
print("FINAL NORMAL COHORT")
print("======================================")

print(
    "Final unique GTEx cortical controls:",
    len(gtex_final)
)

print("\nRegions:")
print(
    gtex_final["SMTSD"]
    .value_counts()
    .to_string()
)

print("\nFinal ischemic-time distribution:")
print(
    gtex_final["SMTSISCH"]
    .describe()
)

print(
    "\nFinal samples >480 min:",
    (gtex_final["SMTSISCH"] > 480).sum()
)


# Save
gtex_final.to_csv(
    OUTDIR /
    "STAGE_A_GTEX_CORTEX_FINAL_QC.csv",
    index=False
)

### CHECK GTEx COVARIATES FOR FINAL 334 CONTROLS

In [ ]:
# ============================================================
# CHECK GTEx COVARIATES FOR FINAL 334 CONTROLS
# ============================================================

cols_to_check = [
    "SMTSISCH",   # ischemic time
    "DTHHRDY",    # Hardy scale / death classification
    "SMRIN",      # RNA integrity number
    "SMGEBTCH"    # sequencing / expression batch
]

print("\n======================================")
print("AVAILABLE COVARIATES")
print("======================================")

for c in cols_to_check:
    print(c, "->", c in gtex_final.columns)


# ============================================================
# ISCHEMIC TIME
# ============================================================

if "SMTSISCH" in gtex_final.columns:

    print("\n======================================")
    print("SMTSISCH")
    print("======================================")

    print(
        pd.to_numeric(
            gtex_final["SMTSISCH"],
            errors="coerce"
        ).describe()
    )


# ============================================================
# HARDY SCALE
# ============================================================

if "DTHHRDY" in gtex_final.columns:

    print("\n======================================")
    print("DTHHRDY")
    print("======================================")

    print(
        gtex_final["DTHHRDY"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )


# ============================================================
# RNA INTEGRITY
# ============================================================

if "SMRIN" in gtex_final.columns:

    print("\n======================================")
    print("SMRIN")
    print("======================================")

    print(
        pd.to_numeric(
            gtex_final["SMRIN"],
            errors="coerce"
        ).describe()
    )


# ============================================================
# BATCH
# ============================================================

if "SMGEBTCH" in gtex_final.columns:

    print("\n======================================")
    print("SMGEBTCH")
    print("======================================")

    print(
        gtex_final["SMGEBTCH"]
        .value_counts(dropna=False)
        .head(30)
        .to_string()
    )

### GTEx COVARIATE RELATIONSHIPS

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# GTEx COVARIATE RELATIONSHIPS
# ============================================================

tmp = gtex_final[
    ["SMTSISCH", "DTHHRDY", "SMRIN"]
].copy()

for c in tmp.columns:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

print("\n======================================")
print("COVARIATE CORRELATIONS")
print("======================================")

print(
    tmp.corr(method="spearman")
    .round(3)
    .to_string()
)

print("\n======================================")
print("ISCHEMIC TIME BY HARDY SCALE")
print("======================================")

print(
    gtex_final
    .assign(
        SMTSISCH=pd.to_numeric(
            gtex_final["SMTSISCH"],
            errors="coerce"
        )
    )
    .groupby("DTHHRDY")["SMTSISCH"]
    .agg(["count", "median", "mean", "std"])
    .round(2)
    .to_string()
)

print("\n======================================")
print("RIN BY HARDY SCALE")
print("======================================")

print(
    gtex_final
    .assign(
        SMRIN=pd.to_numeric(
            gtex_final["SMRIN"],
            errors="coerce"
        )
    )
    .groupby("DTHHRDY")["SMRIN"]
    .agg(["count", "median", "mean", "std"])
    .round(2)
    .to_string()
)

print("\n======================================")
print("GTEx BATCH SUMMARY")
print("======================================")

batch_counts = gtex_final["SMGEBTCH"].value_counts()

print("Number of batches:", batch_counts.size)
print("Median samples per batch:", batch_counts.median())
print("Batches with 1 sample:", (batch_counts == 1).sum())
print("Batches with >=5 samples:", (batch_counts >= 5).sum())
print("Largest batch:", batch_counts.max())